In [3]:
# -*- coding: utf-8 -*-
# 文件名: run_scikit_optimize_search_rf_simple.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import random
import ast
from pathlib import Path

# Set global random seeds for maximum reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# =================================================================================
# Import Scikit-optimize tools
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
# =================================================================================

from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, roc_auc_score, confusion_matrix, roc_curve, accuracy_score, f1_score, recall_score, precision_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import BayesianRidge
from sklearn.base import BaseEstimator, ClassifierMixin, clone

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Configuration ---
N_JOBS_MODEL_INTERNAL = 40
BASE_DIR = Path(os.getcwd())
RFE_LISTS_FOLDER = BASE_DIR / 'RFE_Final_Run/feature_lists'

# 🌟 修改点 1: 指定包含 PostopAKI 的插补文件 🌟
IMPUTED_DATA_FILE = BASE_DIR / 'imputation' / 'imputed_data' / 'train_imputed_random_forest.csv'
TARGET_VARIABLE = 'PostopAKI'

OUTPUT_FOLDER = BASE_DIR / 'Tuning_BayesSearch_Results_Layered'
DATA_OUTPUT_FOLDER = OUTPUT_FOLDER / 'final_training_data'
OUTPUT_PARAMS_EXCEL = OUTPUT_FOLDER / 'best_hyperparameters_bayes.xlsx'
OUTPUT_PERF_EXCEL = OUTPUT_FOLDER / 'model_performance_bayes.xlsx'
OUTPUT_FIGURE_FILE = OUTPUT_FOLDER / 'internal_validation_plot_bayes.png'
N_ITER_BAYESIAN_SEARCH = 50 

# --- Helper Functions (保持不变) ---
def specificity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tn / (tn + fp) if (tn + fp) > 0 else 0.0
def sensitivity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def find_optimal_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden_index = tpr - fpr
    best_idx = np.argmax(youden_index)
    return thresholds[best_idx]

def get_metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

# --- NNET Wrapper (保持不变) ---
class StringMLPClassifier(MLPClassifier):
    def __init__(self, hidden_layer_sizes_str='(100,)', activation='relu', solver='adam', 
                 alpha=0.0001, batch_size='auto', learning_rate='constant', 
                 learning_rate_init=0.001, power_t=0.5, max_iter=500, shuffle=True, 
                 random_state=None, tol=0.0001, verbose=False, warm_start=False, 
                 momentum=0.9, nesterovs_momentum=True, early_stopping=True, 
                 validation_fraction=0.1, beta_1=0.9, beta_2=0.999, epsilon=1e-08, 
                 n_iter_no_change=10, max_fun=15000):
        
        self.hidden_layer_sizes_str = hidden_layer_sizes_str
        super().__init__(
            hidden_layer_sizes=(100,), activation=activation, solver=solver, alpha=alpha, 
            batch_size=batch_size, learning_rate=learning_rate, learning_rate_init=learning_rate_init, 
            power_t=power_t, max_iter=max_iter, shuffle=shuffle, random_state=random_state, 
            tol=tol, verbose=verbose, warm_start=warm_start, momentum=momentum, 
            nesterovs_momentum=nesterovs_momentum, early_stopping=early_stopping, 
            validation_fraction=validation_fraction, beta_1=beta_1, beta_2=beta_2, 
            epsilon=epsilon, n_iter_no_change=n_iter_no_change, max_fun=max_fun
        )

    def fit(self, X, y):
        try:
            parsed_layers = ast.literal_eval(self.hidden_layer_sizes_str)
            if isinstance(parsed_layers, (list, tuple)): self.hidden_layer_sizes = parsed_layers
            else: self.hidden_layer_sizes = (100,)
        except (ValueError, SyntaxError): self.hidden_layer_sizes = (100,)
        return super().fit(X, y)

# --- Main Logic ---
def main():
    for folder in [OUTPUT_FOLDER, DATA_OUTPUT_FOLDER]: folder.mkdir(parents=True, exist_ok=True)

    print("--- Data Loading (Simplified) ---")
    
    # 🌟 1. 直接读取数据 🌟
    print(f"Loading data: {IMPUTED_DATA_FILE.name}")
    try:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='gbk')
    except UnicodeDecodeError:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='utf-8')

    if 'Unnamed: 0' in df_train.columns: df_train.drop(columns=['Unnamed: 0'], inplace=True)

    # 🌟 2. 提取 Target 🌟
    if TARGET_VARIABLE not in df_train.columns:
        raise ValueError(f"CRITICAL ERROR: '{TARGET_VARIABLE}' not found in imputed file.")
    
    y_train = df_train[TARGET_VARIABLE].astype(int)
    print(f"  > Target '{TARGET_VARIABLE}' loaded. Positive rate: {y_train.mean():.4f}")

    # 🌟 3. 特征准备 (白名单机制) 🌟
    preoperative_vars = [ 'Gender', 'Age', 'Height', 'Weight', 'BMI', 'PreopHospitalDays', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'PreopWBC', 'PreopHb', 'PreopAlb', 'PreopALT', 'PreopBUN', 'PreopCr', 'PreopGlucose' ]
    intraoperative_vars = [ 'SurgicalApproach', 'OperationTime', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopBloodLoss', 'IntraopTransfusion', 'IntraopFluid', 'IntraopHES', 'IntraopDextran', 'IntraopGelatin', 'IntraopPlasma', 'IntraopColloid', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage', 'LymphNodesExamined', 'PositiveLymphNodes' ]
    all_predictors_base = preoperative_vars + intraoperative_vars
    categorical_vars_base = ['Gender', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'SurgicalApproach', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopTransfusion', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage']
    
    current_predictors = [v for v in all_predictors_base if v in df_train.columns]
    current_categoricals = [v for v in categorical_vars_base if v in df_train.columns]
    current_continuous = [v for v in current_predictors if v not in current_categoricals]
    
    # 清洗连续变量
    for col in current_continuous:
        df_train[col] = df_train[col].astype(str).str.extract(r'^(\d+[,.]?\d*)', expand=False)
        df_train[col] = df_train[col].str.replace(',', '.', regex=False)
        df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
    
    # 计算衍生指标
    required_cols = ['PLT', 'LYM', 'MONO', 'NE', 'EOS', 'BASO']
    if all(col in df_train.columns for col in required_cols):
        print("  Calculating derived features (NLR, PLR, etc.)...")
        for col in ['LYM']: df_train[col] = df_train[col].replace(0, 1e-6)
        new_features = ['PLR', 'MLR', 'ELR', 'BLR', 'NLR', 'SII']
        df_train['PLR'], df_train['MLR'], df_train['ELR'], df_train['BLR'], df_train['NLR'], df_train['SII'] = \
            df_train['PLT']/df_train['LYM'], df_train['MONO']/df_train['LYM'], df_train['EOS']/df_train['LYM'], \
            df_train['BASO']/df_train['LYM'], df_train['NE']/df_train['LYM'], df_train['PLT']*df_train['NE']/df_train['LYM']
        current_predictors.extend([f for f in new_features if f not in current_predictors])
    
    # 🌟 4. 构建特征矩阵 X 🌟
    # 只包含 current_predictors 列表中的列，绝对安全！
    X_train_full_raw = df_train[current_predictors]
    X_train_full_dummies = pd.get_dummies(X_train_full_raw, columns=current_categoricals, drop_first=True)
    X_train_imputed_full = X_train_full_dummies.copy()
    
    if X_train_imputed_full.isnull().sum().sum() > 0:
        X_train_imputed_full.fillna(X_train_imputed_full.median(), inplace=True)

    print(f"Data ready. Features: {len(X_train_imputed_full.columns)}")

    # --- Models and Spaces ---
    models_and_spaces = {
        'LR': (LogisticRegression(solver='liblinear', random_state=SEED, max_iter=2000, n_jobs=N_JOBS_MODEL_INTERNAL), 
               {'classifier__penalty': Categorical(['l1', 'l2']), 'classifier__C': Real(1e-3, 1e3, prior='log-uniform')}),
        'DT': (DecisionTreeClassifier(random_state=SEED),
               {'classifier__max_depth': Integer(5, 50), 'classifier__min_samples_split': Integer(2, 20)}),
        'RF': (RandomForestClassifier(random_state=SEED, n_jobs=N_JOBS_MODEL_INTERNAL),
               {'classifier__n_estimators': Integer(100, 500), 'classifier__max_depth': Integer(10, 50)}),
        'KNN': (KNeighborsClassifier(n_jobs=N_JOBS_MODEL_INTERNAL),
                {'classifier__n_neighbors': Integer(3, 20), 'classifier__weights': Categorical(['uniform', 'distance'])}),
        'SVM': (SVC(probability=True, random_state=SEED),
                {'classifier__C': Real(1e-2, 1e2, prior='log-uniform'), 'classifier__gamma': Real(1e-4, 1e-1, prior='log-uniform')}),
        'NB': (GaussianNB(),
               {'classifier__var_smoothing': Real(1e-10, 1e-1, prior='log-uniform')}),
        'XGB': (XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED, n_jobs=N_JOBS_MODEL_INTERNAL),
                {'classifier__n_estimators': Integer(100, 400), 'classifier__learning_rate': Real(0.01, 0.3, prior='log-uniform'), 'classifier__max_depth': Integer(3, 10)}),
        'SGBT': (GradientBoostingClassifier(random_state=SEED),
                 {'classifier__n_estimators': Integer(100, 400), 'classifier__learning_rate': Real(0.01, 0.3, prior='log-uniform'), 'classifier__max_depth': Integer(3, 10)}),
        'NNET': (StringMLPClassifier(max_iter=500, random_state=SEED, early_stopping=True), 
                 {'classifier__hidden_layer_sizes_str': Categorical(['(50,)', '(100,)', '(50, 50)', '(100, 50)']), 'classifier__alpha': Real(1e-5, 1e-2, prior='log-uniform')})
    }

    cv_search = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=SEED)
    cv_full = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=SEED)
    
    best_params_list, performance_list = [], []

    for name, (model, space) in models_and_spaces.items():
        print(f"\n--- Processing Model: {name} ---")
        feature_file = RFE_LISTS_FOLDER / f'selected_features_{name}_Combined.txt'
        try:
            with open(feature_file, 'r') as f: 
                selected_features = [line.strip() for line in f if not line.strip().startswith('#') and line.strip()]
        except FileNotFoundError: 
            print(f"  Warning: Feature file not found. Skipping."); continue
            
        valid_features = [f for f in selected_features if f in X_train_imputed_full.columns]
        X_train_model_specific = X_train_imputed_full[valid_features]
        pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])

        if name in ['XGB', 'RF', 'KNN', 'SGBT']:
            search_n_jobs = 1
            n_points_in_batch = 1
        elif name in ['NNET']:
            search_n_jobs = 8
            n_points_in_batch = 2
        else:
            search_n_jobs = 40 
            n_points_in_batch = 4 

        print(f"  Starting BayesSearchCV ({N_ITER_BAYESIAN_SEARCH} trials, n_jobs={search_n_jobs})...")
        bayes_search = BayesSearchCV(
            estimator=pipeline, search_spaces=space, n_iter=N_ITER_BAYESIAN_SEARCH,
            scoring='roc_auc', cv=cv_search, n_jobs=search_n_jobs, n_points=n_points_in_batch,
            verbose=1, random_state=SEED, refit=True, error_score=0.0
        )
        bayes_search.fit(X_train_model_specific, y_train)
        
        best_params_recorded = bayes_search.best_params_.copy()
        if name == 'NNET' and 'classifier__hidden_layer_sizes_str' in best_params_recorded:
            str_val = best_params_recorded.pop('classifier__hidden_layer_sizes_str')
            best_params_recorded['classifier__hidden_layer_sizes'] = str_val
            
        best_params_list.append({'Model': name, 'Best Hyperparameters': str(best_params_recorded)})
        
        # --- 🌟 Manual CV Loop for Optimal Threshold 🌟 ---
        print(f"  Calculating metrics...")
        final_pipeline = bayes_search.best_estimator_
        auc_scores, sens_scores, spec_scores, thresholds = [], [], [], []
        
        for i, (train_idx, val_idx) in enumerate(cv_full.split(X_train_model_specific, y_train)):
            X_train_fold, X_val_fold = X_train_model_specific.iloc[train_idx], X_train_model_specific.iloc[val_idx]
            y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            final_pipeline.fit(X_train_fold, y_train_fold)
            y_prob_val = final_pipeline.predict_proba(X_val_fold)[:, 1]
            best_thresh = find_optimal_threshold(y_val_fold, y_prob_val)
            thresholds.append(best_thresh)
            
            metrics = get_metrics_at_threshold(y_val_fold, y_prob_val, best_thresh)
            auc_scores.append(roc_auc_score(y_val_fold, y_prob_val))
            sens_scores.append(metrics['Sensitivity'])
            spec_scores.append(metrics['Specificity'])
        
        perf_row = {'Model': name}
        perf_row['AUC_Mean'], perf_row['AUC_SD'] = np.mean(auc_scores), np.std(auc_scores)
        perf_row['AUC_95CI_lower'] = perf_row['AUC_Mean'] - 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        perf_row['AUC_95CI_upper'] = perf_row['AUC_Mean'] + 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        perf_row['Sensitivity_Mean'] = np.mean(sens_scores)
        perf_row['Specificity_Mean'] = np.mean(spec_scores)
        perf_row['Avg_Threshold'] = np.mean(thresholds)
        
        performance_list.append(perf_row)
        print(f"  Metrics: AUC={perf_row['AUC_Mean']:.3f}, Sens={perf_row['Sensitivity_Mean']:.3f}, Spec={perf_row['Specificity_Mean']:.3f}")

    # Save
    pd.DataFrame(best_params_list).to_excel(OUTPUT_PARAMS_EXCEL, index=False)
    pd.DataFrame(performance_list).to_excel(OUTPUT_PERF_EXCEL, index=False)
    print(f"\nSaved results to '{OUTPUT_FOLDER}' folder.")
    
    # Plotting
    print("Generating final plot...")
    model_order = ['DT', 'KNN', 'NNET', 'NB', 'SVM', 'SGBT', 'LR', 'RF', 'XGB']
    performance_df = pd.DataFrame(performance_list)
    performance_df['Model'] = pd.Categorical(performance_df['Model'], categories=model_order, ordered=True)
    performance_df.sort_values('Model', inplace=True)
    metrics_to_plot = {'ROC': 'AUC_Mean', 'Spec': 'Specificity_Mean', 'Sens': 'Sensitivity_Mean'}
    fig, axes = plt.subplots(1, 3, figsize=(22, 10))
    for ax, (plot_name, metric_col) in zip(axes, metrics_to_plot.items()):
        vals = performance_df[metric_col]
        y_pos = np.arange(len(performance_df['Model']))
        ax.barh(y_pos, vals, color='skyblue', edgecolor='black', height=0.6)
        ax.set_yticks(y_pos); ax.set_yticklabels(performance_df['Model'], fontsize=12); ax.invert_yaxis()
        ax.set_title(plot_name, fontsize=16); ax.set_xlabel('Value', fontsize=14); ax.set_xlim(0, 1.15)
        for i, v in enumerate(vals):
            try:
                ax.text(v + 0.02, i, f"{v:.3f}", va='center', fontsize=10, fontweight='bold')
            except ValueError: pass
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURE_FILE, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to '{OUTPUT_FIGURE_FILE}'")
    print("\nProcess finished successfully.")

if __name__ == '__main__':
    main()

--- Data Loading (Simplified) ---
Loading data: train_imputed_random_forest.csv
  > Target 'PostopAKI' loaded. Positive rate: 0.0298
Data ready. Features: 15

--- Processing Model: LR ---
  Starting BayesSearchCV (50 trials, n_jobs=40)...
Fitting 5 folds for each of 4 candidates, totalling 20 fits


/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

Fitting 5 folds for each of 4 candidates, totalling 20 fits


/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
  Calculating metrics...
  Metrics: AUC=0.768, Sens=0.765, Spec=0.776

--- Processing Model: DT ---
  Starting BayesSearchCV (50 trials, n_jobs=40)...
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5

...

Fitting 5 folds for each of 4 candidates, totalling 20 fits


/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
  Calculating metrics...
  Metrics: AUC=0.637, Sens=0.679, Spec=0.679

--- Processing Model: XGB ---
  Starting BayesSearchCV (50 trials, n_jobs=1)...
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 fo

/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates

In [ ]:
# -*- coding: utf-8 -*-
# 文件名: run_optuna_search_rf_fixed.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import random
import ast
from pathlib import Path

# Set global random seeds for maximum reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

from optuna.integration import OptunaSearchCV
from optuna.distributions import CategoricalDistribution, FloatDistribution, IntDistribution
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, roc_auc_score, confusion_matrix, roc_curve, accuracy_score, f1_score, recall_score, precision_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import BayesianRidge

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Configuration ---
N_JOBS_MODEL_INTERNAL = 40
BASE_DIR = Path(os.getcwd())
RFE_LISTS_FOLDER = BASE_DIR / 'RFE_Final_Run/feature_lists'

# 🌟 修改点 1: 指定随机森林插补后的文件路径 🌟
IMPUTED_DATA_FILE = BASE_DIR / 'imputation' / 'imputed_data' / 'train_imputed_random_forest.csv'

# 2. 原始数据文件 (包含PostopAKI和ID)
ORIGINAL_DATA_FILE = BASE_DIR / 'inter_eng.csv'

OUTPUT_FOLDER = BASE_DIR / 'Tuning_Optuna_Results'
DATA_OUTPUT_FOLDER = OUTPUT_FOLDER / 'final_training_data'
OUTPUT_PARAMS_EXCEL = OUTPUT_FOLDER / 'best_hyperparameters_optuna.xlsx'
OUTPUT_PERF_EXCEL = OUTPUT_FOLDER / 'model_performance_optuna.xlsx'
OUTPUT_FIGURE_FILE = OUTPUT_FOLDER / 'internal_validation_plot_optuna.png'
TARGET_VARIABLE = 'PostopAKI'
N_TRIALS_OPTUNA = 50 

# --- 辅助函数 ---
def specificity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tn / (tn + fp) if (tn + fp) > 0 else 0.0
def sensitivity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def find_optimal_threshold(y_true, y_prob):
    """Find the threshold that maximizes Youden's Index."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden_index = tpr - fpr
    best_idx = np.argmax(youden_index)
    return thresholds[best_idx]

def get_metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

# --- NNET Wrapper (Fixing string/tuple conversion for Optuna) ---
class StringMLPClassifier(MLPClassifier):
    def __init__(self, hidden_layer_sizes_str='(100,)', activation='relu', solver='adam', 
                 alpha=0.0001, batch_size='auto', learning_rate='constant', 
                 learning_rate_init=0.001, power_t=0.5, max_iter=500, shuffle=True, 
                 random_state=None, tol=0.0001, verbose=False, warm_start=False, 
                 momentum=0.9, nesterovs_momentum=True, early_stopping=True, 
                 validation_fraction=0.1, beta_1=0.9, beta_2=0.999, epsilon=1e-08, 
                 n_iter_no_change=10, max_fun=15000):
        self.hidden_layer_sizes_str = hidden_layer_sizes_str
        super().__init__(
            hidden_layer_sizes=(100,), activation=activation, solver=solver, alpha=alpha, 
            batch_size=batch_size, learning_rate=learning_rate, learning_rate_init=learning_rate_init, 
            power_t=power_t, max_iter=max_iter, shuffle=shuffle, random_state=random_state, 
            tol=tol, verbose=verbose, warm_start=warm_start, momentum=momentum, 
            nesterovs_momentum=nesterovs_momentum, early_stopping=early_stopping, 
            validation_fraction=validation_fraction, beta_1=beta_1, beta_2=beta_2, 
            epsilon=epsilon, n_iter_no_change=n_iter_no_change, max_fun=max_fun
        )
    def fit(self, X, y):
        try:
            # Optuna passes this as a string sometimes, need to parse it back to tuple
            parsed_layers = ast.literal_eval(self.hidden_layer_sizes_str)
            if isinstance(parsed_layers, (list, tuple)): self.hidden_layer_sizes = parsed_layers
            else: self.hidden_layer_sizes = (100,)
        except (ValueError, SyntaxError): self.hidden_layer_sizes = (100,)
        return super().fit(X, y)

# --- Main Logic ---
def main():
    try: os.chdir(BASE_DIR); print(f"Current working directory: {BASE_DIR}")
    except OSError: pass
    for folder in [OUTPUT_FOLDER, DATA_OUTPUT_FOLDER]: Path(folder).mkdir(parents=True, exist_ok=True)

    print("--- Data Loading and Merging (Robust Logic) ---")
    
    # --- 步骤 1: 读取插补后的训练集 (Train Set) ---
    print(f"Loading imputed data: {IMPUTED_DATA_FILE}")
    if not IMPUTED_DATA_FILE.exists():
        print(f"❌ Error: File not found at {IMPUTED_DATA_FILE}")
        return

    try:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='gbk')
    except UnicodeDecodeError:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='utf-8')

    # 🌟 逻辑优化：如果插补文件中已经包含了 PostopAKI，先移除它 🌟
    if TARGET_VARIABLE in df_train.columns:
        print(f"  > Removing existing '{TARGET_VARIABLE}' from imputed file to re-merge cleanly.")
        df_train.drop(columns=[TARGET_VARIABLE], inplace=True)

    # 处理第一列没有列名的情况 (将其重命名为 'ID')
    if df_train.columns[0].startswith('Unnamed'):
        print("  Detected unnamed first column in Train file. Renaming to 'ID'.")
        df_train.rename(columns={df_train.columns[0]: 'ID'}, inplace=True)
    else:
        print(f"  Assuming 1st column '{df_train.columns[0]}' is ID. Renaming to 'ID'.")
        df_train.rename(columns={df_train.columns[0]: 'ID'}, inplace=True)
    
    # --- 步骤 2: 读取原始数据 (Source for Target) ---
    print(f"Loading original data for target mapping: {ORIGINAL_DATA_FILE.name}")
    try:
        df_orig = pd.read_csv(ORIGINAL_DATA_FILE, encoding='gbk')
    except UnicodeDecodeError:
        df_orig = pd.read_csv(ORIGINAL_DATA_FILE, encoding='utf-8')

    # 🌟 修复：如果原始数据没有 ID 列，使用索引创建 ID 🌟
    if 'ID' not in df_orig.columns:
        print(f"  > 'ID' column NOT found in Original file. Creating 'ID' from row index.")
        df_orig['ID'] = df_orig.index
    else:
        print("  > 'ID' column found in Original file. Using it.")
    
    # --- 步骤 3: 合并 PostopAKI ---
    print("  Merging 'PostopAKI' from original data based on 'ID'...")
    
    # 强制 ID 类型一致
    try:
        df_train['ID'] = df_train['ID'].astype(int)
        df_orig['ID'] = df_orig['ID'].astype(int)
    except ValueError:
        print("  > Warning: Non-integer IDs detected. Converting to string.")
        df_train['ID'] = df_train['ID'].astype(str)
        df_orig['ID'] = df_orig['ID'].astype(str)

    if TARGET_VARIABLE not in df_orig.columns:
        raise ValueError(f"Error: Target variable '{TARGET_VARIABLE}' not found in original file.")

    # Left Merge
    df_merged = pd.merge(df_train, df_orig[['ID', TARGET_VARIABLE]], on='ID', how='left')

    # 检查并清除缺失目标变量的行
    if df_merged[TARGET_VARIABLE].isnull().any():
        n_missing = df_merged[TARGET_VARIABLE].isnull().sum()
        print(f"Warning: {n_missing} rows have missing '{TARGET_VARIABLE}' after merge. Dropping these rows.")
        df_merged.dropna(subset=[TARGET_VARIABLE], inplace=True)
    
    df_merged[TARGET_VARIABLE] = df_merged[TARGET_VARIABLE].astype(int)
    y_train = df_merged[TARGET_VARIABLE]
    print(f"  Final Training Set Size: {df_merged.shape[0]} samples. Positive Rate: {y_train.mean():.4f}")

    # --- 步骤 4: 特征准备 ---
    preoperative_vars = [ 'Gender', 'Age', 'Height', 'Weight', 'BMI', 'PreopHospitalDays', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'PreopWBC', 'PreopHb', 'PreopAlb', 'PreopALT', 'PreopBUN', 'PreopCr', 'PreopGlucose' ]
    intraoperative_vars = [ 'SurgicalApproach', 'OperationTime', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopBloodLoss', 'IntraopTransfusion', 'IntraopFluid', 'IntraopHES', 'IntraopDextran', 'IntraopGelatin', 'IntraopPlasma', 'IntraopColloid', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage', 'LymphNodesExamined', 'PositiveLymphNodes' ]
    all_predictors_base = preoperative_vars + intraoperative_vars
    categorical_vars_base = ['Gender', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'SurgicalApproach', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopTransfusion', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage']
    
    current_predictors = [v for v in all_predictors_base if v in df_merged.columns]
    current_categoricals = [v for v in categorical_vars_base if v in df_merged.columns]
    current_continuous = [v for v in current_predictors if v not in current_categoricals]
    
    # 清洗连续变量
    for col in current_continuous:
        df_merged[col] = df_merged[col].astype(str).str.extract(r'^(\d+[,.]?\d*)', expand=False)
        df_merged[col] = df_merged[col].str.replace(',', '.', regex=False)
        df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')
    
    # 计算衍生指标
    required_cols = ['PLT', 'LYM', 'MONO', 'NE', 'EOS', 'BASO']
    if all(col in df_merged.columns for col in required_cols):
        print("  Calculating derived features (NLR, PLR, etc.)...")
        for col in ['LYM']: df_merged[col] = df_merged[col].replace(0, 1e-6)
        new_features = ['PLR', 'MLR', 'ELR', 'BLR', 'NLR', 'SII']
        df_merged['PLR'], df_merged['MLR'], df_merged['ELR'], df_merged['BLR'], df_merged['NLR'], df_merged['SII'] = \
            df_merged['PLT']/df_merged['LYM'], df_merged['MONO']/df_merged['LYM'], df_merged['EOS']/df_merged['LYM'], \
            df_merged['BASO']/df_merged['LYM'], df_merged['NE']/df_merged['LYM'], df_merged['PLT']*df_merged['NE']/df_merged['LYM']
        current_predictors.extend([f for f in new_features if f not in current_predictors])
    
    print("Data preparation complete.")
    
    # 构建特征矩阵
    X_train_full_raw = df_merged[current_predictors]
    current_categoricals = [v for v in current_categoricals if v in X_train_full_raw.columns]
    X_train_full_dummies = pd.get_dummies(X_train_full_raw, columns=current_categoricals, drop_first=True)
    X_train_imputed_full = X_train_full_dummies.copy()
    
    # 填充潜在缺失值
    if X_train_imputed_full.isnull().sum().sum() > 0:
        X_train_imputed_full.fillna(X_train_imputed_full.median(), inplace=True)

    # --- Models and Spaces ---
    models_and_spaces = {
        'LR': (LogisticRegression(solver='liblinear', random_state=SEED, max_iter=2000, n_jobs=N_JOBS_MODEL_INTERNAL), 
               {'classifier__penalty': CategoricalDistribution(['l1', 'l2']), 'classifier__C': FloatDistribution(1e-3, 1e2, log=True)}),
        'DT': (DecisionTreeClassifier(random_state=SEED),
               {'classifier__max_depth': IntDistribution(5, 50), 'classifier__min_samples_split': IntDistribution(2, 20)}),
        'RF': (RandomForestClassifier(random_state=SEED, n_jobs=N_JOBS_MODEL_INTERNAL),
               {'classifier__n_estimators': IntDistribution(100, 500), 'classifier__max_depth': IntDistribution(10, 50)}),
        'KNN': (KNeighborsClassifier(n_jobs=N_JOBS_MODEL_INTERNAL),
                {'classifier__n_neighbors': IntDistribution(3, 20), 'classifier__weights': CategoricalDistribution(['uniform', 'distance'])}),
        'SVM': (SVC(probability=True, random_state=SEED),
                {'classifier__C': FloatDistribution(1e-2, 1e2, log=True), 'classifier__gamma': FloatDistribution(1e-4, 1e-1, log=True)}),
        'NB': (GaussianNB(),
               {'classifier__var_smoothing': FloatDistribution(1e-10, 1e-1, log=True)}),
        'XGB': (XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED, n_jobs=N_JOBS_MODEL_INTERNAL),
                {'classifier__n_estimators': IntDistribution(100, 400), 'classifier__learning_rate': FloatDistribution(0.01, 0.3, log=True), 'classifier__max_depth': IntDistribution(3, 10)}),
        'SGBT': (GradientBoostingClassifier(random_state=SEED),
                 {'classifier__n_estimators': IntDistribution(100, 400), 'classifier__learning_rate': FloatDistribution(0.01, 0.3, log=True), 'classifier__max_depth': IntDistribution(3, 10)}),
        'NNET': (StringMLPClassifier(max_iter=500, random_state=SEED, early_stopping=True), 
                 {'classifier__hidden_layer_sizes_str': CategoricalDistribution(['(50,)', '(100,)', '(50, 50)', '(100, 50)']), 'classifier__alpha': FloatDistribution(1e-5, 1e-2, log=True)})
    }

    cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=SEED)
    best_params_list, performance_list = [], []

    for name, (model, params) in models_and_spaces.items():
        print(f"\n--- Processing Model: {name} ---")
        feature_file = RFE_LISTS_FOLDER / f'selected_features_{name}_Combined.txt'
        try:
            with open(feature_file, 'r') as f: 
                selected_features = [line.strip() for line in f if not line.strip().startswith('#') and line.strip()]
        except FileNotFoundError: 
            print(f"  Warning: Feature file not found at {feature_file}. Skipping.")
            continue
            
        valid_features = [f for f in selected_features if f in X_train_imputed_full.columns]
        if not valid_features:
            print("  Warning: No valid features found for this model.")
            continue
            
        X_train_model_specific = X_train_imputed_full[valid_features]
        pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])

        # --- 动态设置 Optuna 并行度 ---
        if name in ['XGB', 'RF', 'SGBT', 'KNN']:
            search_n_jobs = 1 
        elif name in ['SVM', 'NNET']:
            search_n_jobs = 8
        else: # LR, DT, NB
            search_n_jobs = 40

        print(f"  Starting Optuna Search ({N_TRIALS_OPTUNA} trials, n_jobs={search_n_jobs})...")
        optuna_search = OptunaSearchCV(
            estimator=pipeline, param_distributions=params, n_trials=N_TRIALS_OPTUNA,
            scoring='roc_auc', cv=cv, n_jobs=search_n_jobs, verbose=1, random_state=SEED, refit=True
        )
        optuna_search.fit(X_train_model_specific, y_train)
        
        # --- 结果收集 ---
        best_params_recorded = optuna_search.best_params_.copy()
        if name == 'NNET' and 'classifier__hidden_layer_sizes_str' in best_params_recorded:
            str_val = best_params_recorded.pop('classifier__hidden_layer_sizes_str')
            best_params_recorded['classifier__hidden_layer_sizes'] = str_val
            
        best_params_list.append({'Model': name, 'Best Hyperparameters': str(best_params_recorded)})
        
        # --- 最佳阈值评估 (安全使用单核) ---
        print(f"  Calculating metrics with Optimal Threshold...")
        final_pipeline = optuna_search.best_estimator_
        auc_scores, sens_scores, spec_scores, thresholds = [], [], [], []
        
        # Manual CV loop for stable threshold finding
        for i, (train_idx, val_idx) in enumerate(cv.split(X_train_model_specific, y_train)):
            X_train_fold, X_val_fold = X_train_model_specific.iloc[train_idx], X_train_model_specific.iloc[val_idx]
            y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            final_pipeline.fit(X_train_fold, y_train_fold)
            y_prob_val = final_pipeline.predict_proba(X_val_fold)[:, 1]
            best_thresh = find_optimal_threshold(y_val_fold, y_prob_val)
            thresholds.append(best_thresh)
            
            metrics = get_metrics_at_threshold(y_val_fold, y_prob_val, best_thresh)
            auc_scores.append(roc_auc_score(y_val_fold, y_prob_val))
            sens_scores.append(metrics['Sensitivity'])
            spec_scores.append(metrics['Specificity'])

        perf_row = {'Model': name}
        perf_row['AUC_Mean'], perf_row['AUC_SD'] = np.mean(auc_scores), np.std(auc_scores)
        perf_row['AUC_95CI_lower'] = perf_row['AUC_Mean'] - 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        perf_row['AUC_95CI_upper'] = perf_row['AUC_Mean'] + 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        
        perf_row['Sensitivity_Mean'], perf_row['Sensitivity_SD'] = np.mean(sens_scores), np.std(sens_scores)
        perf_row['Sensitivity_95CI_lower'] = perf_row['Sensitivity_Mean'] - 1.96 * (perf_row['Sensitivity_SD'] / np.sqrt(len(sens_scores)))
        perf_row['Sensitivity_95CI_upper'] = perf_row['Sensitivity_Mean'] + 1.96 * (perf_row['Sensitivity_SD'] / np.sqrt(len(sens_scores)))
        
        perf_row['Specificity_Mean'], perf_row['Specificity_SD'] = np.mean(spec_scores), np.std(spec_scores)
        perf_row['Specificity_95CI_lower'] = perf_row['Specificity_Mean'] - 1.96 * (perf_row['Specificity_SD'] / np.sqrt(len(spec_scores)))
        perf_row['Specificity_95CI_upper'] = perf_row['Specificity_Mean'] + 1.96 * (perf_row['Specificity_SD'] / np.sqrt(len(spec_scores)))
        
        perf_row['Avg_Threshold'] = np.mean(thresholds)
        performance_list.append(perf_row)
        print(f"  Metrics: AUC={perf_row['AUC_Mean']:.3f}, Sens={perf_row['Sensitivity_Mean']:.3f}, Spec={perf_row['Specificity_Mean']:.3f}")

    # --- Save and Plot ---
    params_df = pd.DataFrame(best_params_list)
    performance_df = pd.DataFrame(performance_list)
    params_df.to_excel(OUTPUT_PARAMS_EXCEL, index=False)
    performance_df.to_excel(OUTPUT_PERF_EXCEL, index=False)
    print(f"\nSaved results to '{OUTPUT_FOLDER}' folder.")
    
    print("Generating final plot...")
    model_order = ['DT', 'KNN', 'NNET', 'NB', 'SVM', 'SGBT', 'LR', 'RF', 'XGB']
    performance_df['Model'] = pd.Categorical(performance_df['Model'], categories=model_order, ordered=True)
    performance_df.sort_values('Model', inplace=True)
    metrics_to_plot = {'ROC': 'AUC_Mean', 'Spec': 'Specificity_Mean', 'Sens': 'Sensitivity_Mean'}
    fig, axes = plt.subplots(1, 3, figsize=(22, 10))
    for ax, (plot_name, metric_col) in zip(axes, metrics_to_plot.items()):
        vals = performance_df[metric_col]
        y_pos = np.arange(len(performance_df['Model']))
        ax.barh(y_pos, vals, color='skyblue', edgecolor='black', height=0.6)
        ax.set_yticks(y_pos); ax.set_yticklabels(performance_df['Model'], fontsize=12); ax.invert_yaxis()
        ax.set_title(plot_name, fontsize=16); ax.set_xlabel('Value', fontsize=14); ax.set_xlim(0, 1.15)
        for i, v in enumerate(vals):
            try:
                ax.text(v + 0.02, i, f"{v:.3f}", va='center', fontsize=10, fontweight='bold')
            except ValueError: pass 
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURE_FILE, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to '{OUTPUT_FIGURE_FILE}'")
    print("\nProcess finished successfully.")

if __name__ == '__main__':
    main()

Current working directory: /mnt/d/AKI新分析
--- Data Loading and Merging (Robust Logic) ---
Loading imputed data: /mnt/d/AKI新分析/imputation/imputed_data/train_imputed_random_forest.csv
  > Removing existing 'PostopAKI' from imputed file to re-merge cleanly.
  Detected unnamed first column in Train file. Renaming to 'ID'.
Loading original data for target mapping: inter_eng.csv
  > 'ID' column NOT found in Original file. Creating 'ID' from row index.
  Merging 'PostopAKI' from original data based on 'ID'...
  Final Training Set Size: 2446 samples. Positive Rate: 0.0298
Data preparation complete.

--- Processing Model: LR ---


[I 2025-12-06 21:58:34,581] A new study created in memory with name: no-name-98444d14-7593-497e-a859-258443bd4dd5


  Starting Optuna Search (50 trials, n_jobs=40)...


[I 2025-12-06 21:59:51,969] Trial 3 finished with value: 0.7616385414925465 and parameters: {'classifier__penalty': 'l1', 'classifier__C': 0.027952374434752042}. Best is trial 3 with value: 0.7616385414925465.
[I 2025-12-06 21:59:52,649] Trial 2 finished with value: 0.7676251070048273 and parameters: {'classifier__penalty': 'l2', 'classifier__C': 3.623340809037845}. Best is trial 2 with value: 0.7676251070048273.
[I 2025-12-06 21:59:52,934] Trial 22 finished with value: 0.7569580047208757 and parameters: {'classifier__penalty': 'l1', 'classifier__C': 0.0068844547132225495}. Best is trial 2 with value: 0.7676251070048273.
[I 2025-12-06 21:59:53,608] Trial 21 finished with value: 0.7676303559398444 and parameters: {'classifier__penalty': 'l2', 'classifier__C': 29.328533255773458}. Best is trial 21 with value: 0.7676303559398444.
[I 2025-12-06 21:59:53,629] Trial 1 finished with value: 0.7681073505351508 and parameters: {'classifier__penalty': 'l2', 'classifier__C': 0.012345617248861457}.

  Calculating metrics with Optimal Threshold...


...

[I 2025-12-06 22:01:29,396] Trial 21 finished with value: 0.6624910628808485 and parameters: {'classifier__max_depth': 5, 'classifier__min_samples_split': 14}. Best is trial 21 with value: 0.6624910628808485.
[I 2025-12-06 22:01:29,826] Trial 16 finished with value: 0.661287369378131 and parameters: {'classifier__max_depth': 5, 'classifier__min_samples_split': 18}. Best is trial 21 with value: 0.6624910628808485.
[I 2025-12-06 22:01:30,210] Trial 12 finished with value: 0.5573375552752747 and parameters: {'classifier__max_depth': 31, 'classifier__min_samples_split': 5}. Best is trial 21 with value: 0.6624910628808485.
[I 2025-12-06 22:01:30,229] Trial 20 finished with value: 0.6613058292937428 and parameters: {'classifier__max_depth': 5, 'classifier__min_samples_split': 16}. Best is trial 21 with value: 0.6624910628808485.
[I 2025-12-06 22:01:30,378] Trial 3 finished with value: 0.6406490349304279 and parameters: {'classifier__max_depth': 8, 'classifier__min_samples_split': 8}. Best is

  Calculating metrics with Optimal Threshold...


[I 2025-12-06 22:01:52,561] A new study created in memory with name: no-name-caf9628d-2948-411e-9973-36cc3d327794


  Metrics: AUC=0.662, Sens=0.702, Spec=0.671

--- Processing Model: RF ---
  Starting Optuna Search (50 trials, n_jobs=1)...


[I 2025-12-06 22:03:44,000] Trial 0 finished with value: 0.7843225549713556 and parameters: {'classifier__n_estimators': 282, 'classifier__max_depth': 44}. Best is trial 0 with value: 0.7843225549713556.
[I 2025-12-06 22:04:38,051] Trial 1 finished with value: 0.7792460401882271 and parameters: {'classifier__n_estimators': 127, 'classifier__max_depth': 23}. Best is trial 0 with value: 0.7843225549713556.
[I 2025-12-06 22:06:33,437] Trial 2 finished with value: 0.7847790255342645 and parameters: {'classifier__n_estimators': 294, 'classifier__max_depth': 22}. Best is trial 2 with value: 0.7847790255342645.
[I 2025-12-06 22:07:47,646] Trial 3 finished with value: 0.779471212535647 and parameters: {'classifier__n_estimators': 185, 'classifier__max_depth': 16}. Best is trial 2 with value: 0.7847790255342645.
[I 2025-12-06 22:08:57,856] Trial 4 finished with value: 0.7825333867724303 and parameters: {'classifier__n_estimators': 170, 'classifier__max_depth': 23}. Best is trial 2 with value: 0

In [ ]:
# -*- coding: utf-8 -*-
# 文件名: run_hyperopt_search_rf_fixed.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import random
import ast
from pathlib import Path

# Set global random seeds for maximum reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, roc_auc_score, confusion_matrix, roc_curve, accuracy_score, f1_score, recall_score, precision_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import BayesianRidge

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Configuration ---
# Hyperopt 主要是串行优化，内部模型并行设为 -1 (使用所有核心)
N_JOBS_MODEL_INTERNAL = -1 
BASE_DIR = Path(os.getcwd())
RFE_LISTS_FOLDER = BASE_DIR / 'RFE_Final_Run/feature_lists'

# 🌟 修改点 1: 指定随机森林插补后的文件路径 🌟
IMPUTED_DATA_FILE = BASE_DIR / 'imputation' / 'imputed_data' / 'train_imputed_random_forest.csv'

# 2. 原始数据文件 (包含PostopAKI和ID)
ORIGINAL_DATA_FILE = BASE_DIR / 'inter_eng.csv'

OUTPUT_FOLDER = BASE_DIR / 'Tuning_Hyperopt_Results'
DATA_OUTPUT_FOLDER = OUTPUT_FOLDER / 'final_training_data'
OUTPUT_PARAMS_EXCEL = OUTPUT_FOLDER / 'best_hyperparameters_hyperopt.xlsx'
OUTPUT_PERF_EXCEL = OUTPUT_FOLDER / 'model_performance_hyperopt.xlsx'
OUTPUT_FIGURE_FILE = OUTPUT_FOLDER / 'internal_validation_plot_hyperopt.png'
TARGET_VARIABLE = 'PostopAKI'
MAX_EVALS_HYPEROPT = 50

# --- 辅助函数 ---
def specificity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tn / (tn + fp) if (tn + fp) > 0 else 0.0
def sensitivity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def find_optimal_threshold(y_true, y_prob):
    """Find the threshold that maximizes Youden's Index."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden_index = tpr - fpr
    best_idx = np.argmax(youden_index)
    return thresholds[best_idx]

def get_metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

# --- Main Logic ---
def main():
    try: os.chdir(BASE_DIR); print(f"Current working directory: {BASE_DIR}")
    except OSError: pass
    for folder in [OUTPUT_FOLDER, DATA_OUTPUT_FOLDER]: Path(folder).mkdir(parents=True, exist_ok=True)

    print("--- Data Loading and Merging (Robust Logic) ---")
    
    # --- 步骤 1: 读取插补后的训练集 (Train Set) ---
    print(f"Loading imputed data: {IMPUTED_DATA_FILE}")
    if not IMPUTED_DATA_FILE.exists():
        print(f"❌ Error: File not found at {IMPUTED_DATA_FILE}")
        return

    try:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='gbk')
    except UnicodeDecodeError:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='utf-8')

    # 🌟 逻辑优化：如果插补文件中已经包含了 PostopAKI，先移除它 🌟
    if TARGET_VARIABLE in df_train.columns:
        print(f"  > Removing existing '{TARGET_VARIABLE}' from imputed file to re-merge cleanly.")
        df_train.drop(columns=[TARGET_VARIABLE], inplace=True)

    # 处理第一列没有列名的情况 (将其重命名为 'ID')
    if df_train.columns[0].startswith('Unnamed'):
        print("  Detected unnamed first column in Train file. Renaming to 'ID'.")
        df_train.rename(columns={df_train.columns[0]: 'ID'}, inplace=True)
    else:
        print(f"  Assuming 1st column '{df_train.columns[0]}' is ID. Renaming to 'ID'.")
        df_train.rename(columns={df_train.columns[0]: 'ID'}, inplace=True)
    
    # --- 步骤 2: 读取原始数据 (Source for Target) ---
    print(f"Loading original data for target mapping: {ORIGINAL_DATA_FILE.name}")
    try:
        df_orig = pd.read_csv(ORIGINAL_DATA_FILE, encoding='gbk')
    except UnicodeDecodeError:
        df_orig = pd.read_csv(ORIGINAL_DATA_FILE, encoding='utf-8')

    # 🌟 修复：如果原始数据没有 ID 列，使用索引创建 ID 🌟
    if 'ID' not in df_orig.columns:
        print(f"  > 'ID' column NOT found in Original file. Creating 'ID' from row index.")
        df_orig['ID'] = df_orig.index
    else:
        print("  > 'ID' column found in Original file. Using it.")
    
    # --- 步骤 3: 合并 PostopAKI ---
    print("  Merging 'PostopAKI' from original data based on 'ID'...")
    
    # 强制 ID 类型一致
    try:
        df_train['ID'] = df_train['ID'].astype(int)
        df_orig['ID'] = df_orig['ID'].astype(int)
    except ValueError:
        print("  > Warning: Non-integer IDs detected. Converting to string.")
        df_train['ID'] = df_train['ID'].astype(str)
        df_orig['ID'] = df_orig['ID'].astype(str)

    if TARGET_VARIABLE not in df_orig.columns:
        raise ValueError(f"Error: Target variable '{TARGET_VARIABLE}' not found in original file.")

    # Left Merge
    df_merged = pd.merge(df_train, df_orig[['ID', TARGET_VARIABLE]], on='ID', how='left')

    # 检查并清除缺失目标变量的行
    if df_merged[TARGET_VARIABLE].isnull().any():
        n_missing = df_merged[TARGET_VARIABLE].isnull().sum()
        print(f"Warning: {n_missing} rows have missing '{TARGET_VARIABLE}' after merge. Dropping these rows.")
        df_merged.dropna(subset=[TARGET_VARIABLE], inplace=True)
    
    df_merged[TARGET_VARIABLE] = df_merged[TARGET_VARIABLE].astype(int)
    y_train = df_merged[TARGET_VARIABLE]
    print(f"  Final Training Set Size: {df_merged.shape[0]} samples. Positive Rate: {y_train.mean():.4f}")

    # --- 步骤 4: 特征准备 ---
    preoperative_vars = [ 'Gender', 'Age', 'Height', 'Weight', 'BMI', 'PreopHospitalDays', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'PreopWBC', 'PreopHb', 'PreopAlb', 'PreopALT', 'PreopBUN', 'PreopCr', 'PreopGlucose' ]
    intraoperative_vars = [ 'SurgicalApproach', 'OperationTime', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopBloodLoss', 'IntraopTransfusion', 'IntraopFluid', 'IntraopHES', 'IntraopDextran', 'IntraopGelatin', 'IntraopPlasma', 'IntraopColloid', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage', 'LymphNodesExamined', 'PositiveLymphNodes' ]
    all_predictors_base = preoperative_vars + intraoperative_vars
    categorical_vars_base = ['Gender', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'SurgicalApproach', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopTransfusion', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage']
    
    current_predictors = [v for v in all_predictors_base if v in df_merged.columns]
    current_categoricals = [v for v in categorical_vars_base if v in df_merged.columns]
    current_continuous = [v for v in current_predictors if v not in current_categoricals]
    
    # 清洗连续变量
    for col in current_continuous:
        df_merged[col] = df_merged[col].astype(str).str.extract(r'^(\d+[,.]?\d*)', expand=False)
        df_merged[col] = df_merged[col].str.replace(',', '.', regex=False)
        df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')
    
    # 计算衍生指标
    required_cols = ['PLT', 'LYM', 'MONO', 'NE', 'EOS', 'BASO']
    if all(col in df_merged.columns for col in required_cols):
        print("  Calculating derived features (NLR, PLR, etc.)...")
        for col in ['LYM']: df_merged[col] = df_merged[col].replace(0, 1e-6)
        new_features = ['PLR', 'MLR', 'ELR', 'BLR', 'NLR', 'SII']
        df_merged['PLR'], df_merged['MLR'], df_merged['ELR'], df_merged['BLR'], df_merged['NLR'], df_merged['SII'] = \
            df_merged['PLT']/df_merged['LYM'], df_merged['MONO']/df_merged['LYM'], df_merged['EOS']/df_merged['LYM'], \
            df_merged['BASO']/df_merged['LYM'], df_merged['NE']/df_merged['LYM'], df_merged['PLT']*df_merged['NE']/df_merged['LYM']
        current_predictors.extend([f for f in new_features if f not in current_predictors])
    
    print("Data preparation complete.")
    
    # 构建特征矩阵
    X_train_full_raw = df_merged[current_predictors]
    current_categoricals = [v for v in current_categoricals if v in X_train_full_raw.columns]
    X_train_full_dummies = pd.get_dummies(X_train_full_raw, columns=current_categoricals, drop_first=True)
    X_train_imputed_full = X_train_full_dummies.copy()
    
    # 填充潜在缺失值
    if X_train_imputed_full.isnull().sum().sum() > 0:
        X_train_imputed_full.fillna(X_train_imputed_full.median(), inplace=True)

    # --- Define Models and Hyperparameter Search Spaces for Hyperopt ---
    models_and_spaces = {
        'LR': (LogisticRegression(solver='liblinear', random_state=SEED, max_iter=2000), 
               {'classifier__penalty': hp.choice('lr_penalty', ['l1', 'l2']), 'classifier__C': hp.loguniform('lr_C', np.log(0.001), np.log(100))}),
        'DT': (DecisionTreeClassifier(random_state=SEED),
               {'classifier__max_depth': hp.choice('dt_max_depth', [None] + list(range(5, 50))), 'classifier__min_samples_split': hp.quniform('dt_min_samples_split', 2, 20, 1)}),
        'RF': (RandomForestClassifier(random_state=SEED),
               {'classifier__n_estimators': hp.quniform('rf_n_estimators', 100, 500, 50), 'classifier__max_depth': hp.choice('rf_max_depth', [None] + list(range(10, 60, 10)))}),
        'KNN': (KNeighborsClassifier(),
                {'classifier__n_neighbors': hp.quniform('knn_n_neighbors', 3, 25, 2), 'classifier__metric': hp.choice('knn_metric', ['euclidean', 'manhattan'])}),
        'SVM': (SVC(probability=True, random_state=SEED),
                {'classifier__C': hp.loguniform('svm_C', np.log(0.1), np.log(100)), 'classifier__gamma': hp.loguniform('svm_gamma', np.log(0.0001), np.log(0.1))}),
        'NB': (GaussianNB(),
               {'classifier__var_smoothing': hp.loguniform('nb_var_smoothing', np.log(1e-10), np.log(1e-1))}),
        'XGB': (XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED),
                {'classifier__n_estimators': hp.quniform('xgb_n_estimators', 100, 400, 50), 'classifier__learning_rate': hp.loguniform('xgb_learning_rate', np.log(0.01), np.log(0.3)), 'classifier__max_depth': hp.quniform('xgb_max_depth', 3, 10, 1)}),
        'SGBT': (GradientBoostingClassifier(random_state=SEED),
                 {'classifier__n_estimators': hp.quniform('sgbt_n_estimators', 100, 400, 50), 'classifier__learning_rate': hp.loguniform('sgbt_learning_rate', np.log(0.01), np.log(0.3)), 'classifier__max_depth': hp.quniform('sgbt_max_depth', 3, 10, 1)}),
        'NNET': (MLPClassifier(max_iter=500, random_state=SEED, early_stopping=True),
                 {'classifier__hidden_layer_sizes': hp.choice('nnet_hidden_layer_sizes', [(50,), (100,), (50, 50), (100, 50)]), 'classifier__alpha': hp.loguniform('nnet_alpha', np.log(1e-5), np.log(1e-2))})
    }

    cv_full = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=SEED)
    best_params_list, performance_list = [], []
    current_n_jobs = -1

    for name, (model, space) in models_and_spaces.items():
        print(f"\n--- Processing Model: {name} ---")
        feature_file_path = RFE_LISTS_FOLDER / f'selected_features_{name}_Combined.txt'
        try:
            with open(feature_file_path, 'r') as f: selected_features = [line.strip() for line in f if not line.strip().startswith('#') and line.strip()]
        except FileNotFoundError: print(f"  Warning: Feature file not found. Skipping."); continue
            
        valid_features = [f for f in selected_features if f in X_train_imputed_full.columns]
        X_train_model_specific = X_train_imputed_full[valid_features]
        
        pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])

        # Define objective for Hyperopt
        def objective(params):
            # Hyperopt returns floats for quniform, sklearn needs ints for these params
            # 🌟 Fix: Ensure params are converted to int for tree-based models and neighbors 🌟
            for param in ['classifier__n_estimators', 'classifier__max_depth', 'classifier__min_child_weight', 'classifier__n_neighbors', 'classifier__min_samples_split']:
                if param in params and params[param] is not None:
                    params[param] = int(params[param])
            
            pipeline.set_params(**params)
            # Use 5-fold CV for speed during search
            score = cross_val_score(pipeline, X_train_model_specific, y_train, scoring='roc_auc', cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=SEED), n_jobs=-1).mean()
            return {'loss': 1 - score, 'status': STATUS_OK}

        print(f"  Starting Hyperopt Search for {name} ({MAX_EVALS_HYPEROPT} evals)...")
        trials = Trials()
        best_params_raw = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=MAX_EVALS_HYPEROPT, trials=trials, rstate=np.random.default_rng(SEED))
        best_params = space_eval(space, best_params_raw)
        
        # Clean params (convert back to int for final usage)
        for param in ['classifier__n_estimators', 'classifier__max_depth', 'classifier__n_neighbors', 'classifier__min_samples_split']:
            if param in best_params and best_params[param] is not None: 
                best_params[param] = int(best_params[param])
            
        best_params_list.append({'Model': name, 'Best Hyperparameters': str(best_params)})
        
        # --- 🌟 CRITICAL: Find Optimal Threshold using Cross-Validation ---
        print(f"  Calculating performance metrics with Optimal Threshold...")
        final_pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])
        final_pipeline.set_params(**best_params)
        
        auc_scores, sens_scores, spec_scores, thresholds = [], [], [], []
        
        # Manual CV loop for stable metric calculation
        for i, (train_idx, val_idx) in enumerate(cv_full.split(X_train_model_specific, y_train)):
            X_train_fold, X_val_fold = X_train_model_specific.iloc[train_idx], X_train_model_specific.iloc[val_idx]
            y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            final_pipeline.fit(X_train_fold, y_train_fold)
            y_prob_val = final_pipeline.predict_proba(X_val_fold)[:, 1]
            best_thresh = find_optimal_threshold(y_val_fold, y_prob_val)
            thresholds.append(best_thresh)
            
            metrics = get_metrics_at_threshold(y_val_fold, y_prob_val, best_thresh)
            auc_scores.append(roc_auc_score(y_val_fold, y_prob_val))
            sens_scores.append(metrics['Sensitivity'])
            spec_scores.append(metrics['Specificity'])
        
        perf_row = {'Model': name}
        perf_row['AUC_Mean'], perf_row['AUC_SD'] = np.mean(auc_scores), np.std(auc_scores)
        perf_row['AUC_95CI_lower'] = perf_row['AUC_Mean'] - 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        perf_row['AUC_95CI_upper'] = perf_row['AUC_Mean'] + 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        
        perf_row['Sensitivity_Mean'], perf_row['Sensitivity_SD'] = np.mean(sens_scores), np.std(sens_scores)
        perf_row['Sensitivity_95CI_lower'] = perf_row['Sensitivity_Mean'] - 1.96 * (perf_row['Sensitivity_SD'] / np.sqrt(len(sens_scores)))
        perf_row['Sensitivity_95CI_upper'] = perf_row['Sensitivity_Mean'] + 1.96 * (perf_row['Sensitivity_SD'] / np.sqrt(len(sens_scores)))
        
        perf_row['Specificity_Mean'], perf_row['Specificity_SD'] = np.mean(spec_scores), np.std(spec_scores)
        perf_row['Specificity_95CI_lower'] = perf_row['Specificity_Mean'] - 1.96 * (perf_row['Specificity_SD'] / np.sqrt(len(spec_scores)))
        perf_row['Specificity_95CI_upper'] = perf_row['Specificity_Mean'] + 1.96 * (perf_row['Specificity_SD'] / np.sqrt(len(spec_scores)))
        
        perf_row['Avg_Threshold'] = np.mean(thresholds)
        performance_list.append(perf_row)
        print(f"  Metrics: AUC={perf_row['AUC_Mean']:.3f}, Sens={perf_row['Sensitivity_Mean']:.3f}, Spec={perf_row['Specificity_Mean']:.3f}")

    # --- Save and Plot Final Results ---
    params_df = pd.DataFrame(best_params_list)
    performance_df = pd.DataFrame(performance_list)
    params_df.to_excel(OUTPUT_PARAMS_EXCEL, index=False)
    performance_df.to_excel(OUTPUT_PERF_EXCEL, index=False)
    print(f"\nSaved results to '{OUTPUT_FOLDER}' folder.")
    
    # 绘图逻辑
    print("Generating final plot...")
    model_order = ['DT', 'KNN', 'NNET', 'NB', 'SVM', 'SGBT', 'LR', 'RF', 'XGB']
    performance_df['Model'] = pd.Categorical(performance_df['Model'], categories=model_order, ordered=True)
    performance_df.sort_values('Model', inplace=True)
    metrics_to_plot = {'ROC': 'AUC', 'Spec': 'Specificity', 'Sens': 'Sensitivity'}
    fig, axes = plt.subplots(1, 3, figsize=(22, 10))
    for ax, (plot_name, metric_base) in zip(axes, metrics_to_plot.items()):
        means, sds = performance_df[f'{metric_base}_Mean'], performance_df[f'{metric_base}_SD']
        ci_lowers, ci_uppers = performance_df[f'{metric_base}_95CI_lower'], performance_df[f'{metric_base}_95CI_upper']
        y_pos = np.arange(len(performance_df['Model']))
        ci_error, sd_error = [means - ci_lowers, ci_uppers - means], sds
        ax.errorbar(means, y_pos, xerr=ci_error, fmt='none', ecolor='skyblue', elinewidth=10, capsize=0)
        ax.errorbar(means, y_pos, xerr=sd_error, fmt='none', ecolor='dimgray', elinewidth=1.5, capsize=4)
        ax.plot(means, y_pos, 'o', color='black', markersize=6)
        ax.set_yticks(y_pos); ax.set_yticklabels(performance_df['Model'], fontsize=12); ax.invert_yaxis()
        ax.set_title(plot_name, fontsize=16); ax.set_xlabel('Value', fontsize=14)
        ax.set_xlim(0, 1.2); ax.grid(axis='x', linestyle='--', alpha=0.6)
        for i in range(len(performance_df)):
            if i < len(performance_df) and i < len(means):
                mean_val, sd_val = means.iloc[i], sds.iloc[i]
                ci_lower_val, ci_upper_val = ci_lowers.iloc[i], ci_uppers.iloc[i]
                try:
                    ax.text(1.03, y_pos[i] + 0.18, f"{mean_val:.3f}±{sd_val:.3f}", va='center', ha='left', fontsize=11)
                    ax.text(1.03, y_pos[i] - 0.18, f"95% CI ({ci_lower_val:.4f}‒{ci_upper_val:.4f})", va='center', ha='left', fontsize=11)
                except Exception: pass
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURE_FILE, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to '{OUTPUT_FIGURE_FILE}'")
    print("\nProcess finished successfully.")

if __name__ == '__main__':
    main()

In [6]:
# -*- coding: utf-8 -*-
# 文件名: run_twostage_search_rf_simple.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import random
import ast
from pathlib import Path

# Set global random seeds for maximum reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, roc_auc_score, confusion_matrix, roc_curve, accuracy_score, f1_score, recall_score, precision_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import BayesianRidge
from sklearn.base import BaseEstimator, ClassifierMixin

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 路径配置 ---
BASE_DIR = Path(os.getcwd())
RFE_LISTS_FOLDER = BASE_DIR / 'RFE_Final_Run/feature_lists'

# 🌟 修改点 1: 指定包含 PostopAKI 的插补文件 🌟
IMPUTED_DATA_FILE = BASE_DIR / 'imputation' / 'imputed_data' / 'train_imputed_random_forest.csv'
TARGET_VARIABLE = 'PostopAKI'

OUTPUT_FOLDER = BASE_DIR / 'Tuning_TwoStage_Results'
DATA_OUTPUT_FOLDER = OUTPUT_FOLDER / 'final_training_data'
OUTPUT_PARAMS_EXCEL = OUTPUT_FOLDER / 'best_hyperparameters_twostage.xlsx'
OUTPUT_PERF_EXCEL = OUTPUT_FOLDER / 'model_performance_twostage.xlsx'
OUTPUT_FIGURE_FILE = OUTPUT_FOLDER / 'internal_validation_plot_twostage.png'
N_ITER_BAYESIAN_SEARCH = 50 

# --- 辅助函数 ---
def specificity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tn / (tn + fp) if (tn + fp) > 0 else 0.0
def sensitivity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def find_optimal_threshold(y_true, y_prob):
    """Find the threshold that maximizes Youden's Index."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden_index = tpr - fpr
    best_idx = np.argmax(youden_index)
    return thresholds[best_idx]

def get_metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

# --- NNET 包装器 ---
class StringMLPClassifier(MLPClassifier):
    def __init__(self, hidden_layer_sizes_str='(100,)', activation='relu', solver='adam', 
                 alpha=0.0001, batch_size='auto', learning_rate='constant', 
                 learning_rate_init=0.001, power_t=0.5, max_iter=500, shuffle=True, 
                 random_state=None, tol=0.0001, verbose=False, warm_start=False, 
                 momentum=0.9, nesterovs_momentum=True, early_stopping=True, 
                 validation_fraction=0.1, beta_1=0.9, beta_2=0.999, epsilon=1e-08, 
                 n_iter_no_change=10, max_fun=15000):
        self.hidden_layer_sizes_str = hidden_layer_sizes_str
        super().__init__(
            hidden_layer_sizes=(100,), activation=activation, solver=solver, alpha=alpha, 
            batch_size=batch_size, learning_rate=learning_rate, learning_rate_init=learning_rate_init, 
            power_t=power_t, max_iter=max_iter, shuffle=shuffle, random_state=random_state, 
            tol=tol, verbose=verbose, warm_start=warm_start, momentum=momentum, 
            nesterovs_momentum=nesterovs_momentum, early_stopping=early_stopping, 
            validation_fraction=validation_fraction, beta_1=beta_1, beta_2=beta_2, 
            epsilon=epsilon, n_iter_no_change=n_iter_no_change, max_fun=max_fun
        )
    def fit(self, X, y):
        try:
            parsed_layers = ast.literal_eval(self.hidden_layer_sizes_str)
            if isinstance(parsed_layers, (list, tuple)): self.hidden_layer_sizes = parsed_layers
            else: self.hidden_layer_sizes = (100,)
        except (ValueError, SyntaxError): self.hidden_layer_sizes = (100,)
        return super().fit(X, y)

# --- 主逻辑 ---
def main():
    try: os.chdir(BASE_DIR); print(f"Current working directory: {BASE_DIR}")
    except OSError: pass
    for folder in [OUTPUT_FOLDER, DATA_OUTPUT_FOLDER]: Path(folder).mkdir(parents=True, exist_ok=True)
    
    print("--- Data Loading (Simplified) ---")
    
    # 🌟 1. 直接读取数据 🌟
    print(f"Loading data: {IMPUTED_DATA_FILE.name}")
    try:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='gbk')
    except UnicodeDecodeError:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='utf-8')

    # 清洗可能存在的无名索引列
    if 'Unnamed: 0' in df_train.columns: df_train.drop(columns=['Unnamed: 0'], inplace=True)
    
    # 🌟 2. 提取 Target 🌟
    if TARGET_VARIABLE not in df_train.columns:
        raise ValueError(f"CRITICAL ERROR: '{TARGET_VARIABLE}' not found in imputed file.")
    
    y_train = df_train[TARGET_VARIABLE].astype(int)
    print(f"  > Target '{TARGET_VARIABLE}' loaded. Positive rate: {y_train.mean():.4f}")

    # 🌟 3. 特征准备 (白名单机制，排除 ID/Target) 🌟
    preoperative_vars = [ 'Gender', 'Age', 'Height', 'Weight', 'BMI', 'PreopHospitalDays', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'PreopWBC', 'PreopHb', 'PreopAlb', 'PreopALT', 'PreopBUN', 'PreopCr', 'PreopGlucose' ]
    intraoperative_vars = [ 'SurgicalApproach', 'OperationTime', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopBloodLoss', 'IntraopTransfusion', 'IntraopFluid', 'IntraopHES', 'IntraopDextran', 'IntraopGelatin', 'IntraopPlasma', 'IntraopColloid', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage', 'LymphNodesExamined', 'PositiveLymphNodes' ]
    all_predictors_base = preoperative_vars + intraoperative_vars
    categorical_vars_base = ['Gender', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'SurgicalApproach', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopTransfusion', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage']
    
    current_predictors = [v for v in all_predictors_base if v in df_train.columns]
    current_categoricals = [v for v in categorical_vars_base if v in df_train.columns]
    current_continuous = [v for v in current_predictors if v not in current_categoricals]
    
    # 清洗连续变量
    for col in current_continuous:
        df_train[col] = df_train[col].astype(str).str.extract(r'^(\d+[,.]?\d*)', expand=False)
        df_train[col] = df_train[col].str.replace(',', '.', regex=False)
        df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
        
    # 计算衍生指标
    required_cols = ['PLT', 'LYM', 'MONO', 'NE', 'EOS', 'BASO']
    if all(col in df_train.columns for col in required_cols):
        print("  Calculating derived features (NLR, PLR, etc.)...")
        for col in ['LYM']: df_train[col] = df_train[col].replace(0, 1e-6)
        new_features = ['PLR', 'MLR', 'ELR', 'BLR', 'NLR', 'SII']
        df_train['PLR'], df_train['MLR'], df_train['ELR'], df_train['BLR'], df_train['NLR'], df_train['SII'] = \
            df_train['PLT']/df_train['LYM'], df_train['MONO']/df_train['LYM'], df_train['EOS']/df_train['LYM'], \
            df_train['BASO']/df_train['LYM'], df_train['NE']/df_train['LYM'], df_train['PLT']*df_train['NE']/df_train['LYM']
        current_predictors.extend([f for f in new_features if f not in current_predictors])
    
    # 🌟 4. 构建特征矩阵 X 🌟
    X_train_full_raw = df_train[current_predictors]
    X_train_full_dummies = pd.get_dummies(X_train_full_raw, columns=current_categoricals, drop_first=True)
    X_train_imputed_full = X_train_full_dummies.copy()
    
    if X_train_imputed_full.isnull().sum().sum() > 0:
        X_train_imputed_full.fillna(X_train_imputed_full.median(), inplace=True)

    print(f"Data ready. Features: {len(X_train_imputed_full.columns)}, Samples: {len(X_train_imputed_full)}")
    
    # --- Models and Spaces ---
    models = {
        'LR': LogisticRegression(solver='liblinear', random_state=SEED, max_iter=2000), 'DT': DecisionTreeClassifier(random_state=SEED),
        'RF': RandomForestClassifier(random_state=SEED), 'KNN': KNeighborsClassifier(),
        'SVM': SVC(probability=True, random_state=SEED, kernel='rbf'), 'NB': GaussianNB(),
        'XGB': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED),
        'SGBT': GradientBoostingClassifier(random_state=SEED),
        'NNET': StringMLPClassifier(max_iter=500, random_state=SEED, early_stopping=True)  
    }
    coarse_params = {
        'LR': {'classifier__C': [0.01, 1.0, 100.0]}, 'DT': {'classifier__max_depth': [5, 20, None]},
        'RF': {'classifier__n_estimators': [100, 300], 'classifier__max_depth': [10, 30]},
        'KNN': {'classifier__n_neighbors': [3, 15]}, 'SVM': {'classifier__C': [0.1, 10], 'classifier__gamma': [0.001, 0.1]},
        'NB': {}, 'XGB': {'classifier__learning_rate': [0.01, 0.2], 'classifier__max_depth': [3, 8]},
        'SGBT': {'classifier__learning_rate': [0.01, 0.2], 'classifier__max_depth': [3, 8]},
        'NNET': {'classifier__hidden_layer_sizes_str': ['(50,)', '(100,)']}  
    }
    hidden_layer_tuples = [(i,) for i in range(50, 121, 10)] 
    focused_spaces = {
        'LR': {'classifier__C': Real(1e-3, 1e3, prior='log-uniform')},
        'DT': {'classifier__max_depth': Categorical([5, 10, 15, 20, 30, None]), 'classifier__min_samples_split': Integer(2, 20)},
        'RF': {'classifier__n_estimators': Integer(100, 500), 'classifier__max_depth': Categorical([10, 20, 30, None])},
        'KNN': {'classifier__n_neighbors': Integer(3, 25)},
        'SVM': {'classifier__C': Real(1e-2, 1e2, prior='log-uniform'), 'classifier__gamma': Real(1e-4, 1e-1, prior='log-uniform')},
        'NB': {'classifier__var_smoothing': Real(1e-10, 1e-1, prior='log-uniform')},
        'XGB': {'classifier__n_estimators': Integer(100, 400), 'classifier__learning_rate': Real(0.01, 0.3, prior='log-uniform'), 'classifier__max_depth': Integer(3, 10)},
        'SGBT': {'classifier__n_estimators': Integer(100, 400), 'classifier__learning_rate': Real(0.01, 0.3, prior='log-uniform'), 'classifier__max_depth': Integer(3, 10)},
        'NNET': {'classifier__hidden_layer_sizes_str': Categorical([str(t) for t in hidden_layer_tuples]), 'classifier__alpha': Real(1e-5, 1e-2, prior='log-uniform')}
    }
    
    cv_full = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=SEED)
    scoring = {'AUC': 'roc_auc'} 
    best_params_list, performance_list = [], []
    current_n_jobs = -1 

    for name, model in models.items():
        print(f"\n--- Processing Model: {name} ---")
        feature_file_path = RFE_LISTS_FOLDER / f'selected_features_{name}_Combined.txt'
        try:
            with open(feature_file_path, 'r') as f: selected_features = [line.strip() for line in f if not line.strip().startswith('#') and line.strip()]
        except FileNotFoundError: 
            print(f"  Warning: Feature file not found at {feature_file_path}. Skipping."); continue
            
        valid_features = [f for f in selected_features if f in X_train_imputed_full.columns]
        if not valid_features:
            print("  Warning: No valid features found for this model.")
            continue
            
        X_train_model_specific = X_train_imputed_full[valid_features]
        initial_pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])
        current_pipeline = initial_pipeline
        best_coarse_params = {}
        
        # --- STAGE 1: Coarse Grid Search ---
        if coarse_params.get(name):
            print(f"\n--- STAGE 1: Coarse Grid Search for {name} ---")
            grid_search = GridSearchCV(estimator=initial_pipeline, param_grid=coarse_params[name], scoring='roc_auc', cv=5, n_jobs=current_n_jobs, verbose=1)
            grid_search.fit(X_train_model_specific, y_train)
            print(f"  Coarse search best params: {grid_search.best_params_}")
            best_coarse_params = grid_search.best_params_
            current_pipeline = grid_search.best_estimator_
        else:
            print(f"\n--- STAGE 1: Skipping Coarse Grid Search for {name} ---")

        # --- STAGE 2: Focused Bayesian Search ---
        print(f"\n--- STAGE 2: Focused Bayesian Search for {name} ---")
        stage2_spaces = focused_spaces[name].copy()
        if name == 'NNET' and 'classifier__hidden_layer_sizes_str' in best_coarse_params:
            stage2_spaces.pop('classifier__hidden_layer_sizes_str', None)
        
        bayes_search = BayesSearchCV(
            estimator=current_pipeline, search_spaces=stage2_spaces, n_iter=N_ITER_BAYESIAN_SEARCH,
            scoring=scoring, refit='AUC', cv=cv_full, n_jobs=current_n_jobs, verbose=1, random_state=SEED
        )
        bayes_search.fit(X_train_model_specific, y_train)
        
        best_params_recorded = bayes_search.best_params_.copy()
        if name == 'NNET' and 'classifier__hidden_layer_sizes_str' in best_coarse_params:
             best_params_recorded['classifier__hidden_layer_sizes_str'] = best_coarse_params['classifier__hidden_layer_sizes_str']
        best_params_list.append({'Model': name, 'Best Hyperparameters': str(best_params_recorded)})
        
        # --- 🌟 CRITICAL: Find Optimal Threshold using Cross-Validation (FIXED) 🌟 ---
        print(f"  Calculating performance metrics with Optimal Threshold...")
        final_pipeline = bayes_search.best_estimator_
        
        auc_scores, sens_scores, spec_scores, thresholds = [], [], [], []
        
        # Manual CV loop to find threshold per fold (avoiding cross_val_predict partition error)
        for i, (train_idx, val_idx) in enumerate(cv_full.split(X_train_model_specific, y_train)):
            X_train_fold, X_val_fold = X_train_model_specific.iloc[train_idx], X_train_model_specific.iloc[val_idx]
            y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            # Fit on train fold
            final_pipeline.fit(X_train_fold, y_train_fold)
            
            # Predict on validation fold
            y_prob_val = final_pipeline.predict_proba(X_val_fold)[:, 1]
            
            # Find optimal threshold for this fold
            best_thresh = find_optimal_threshold(y_val_fold, y_prob_val)
            thresholds.append(best_thresh)
            
            # Calculate metrics
            metrics = get_metrics_at_threshold(y_val_fold, y_prob_val, best_thresh)
            auc_scores.append(roc_auc_score(y_val_fold, y_prob_val))
            sens_scores.append(metrics['Sensitivity'])
            spec_scores.append(metrics['Specificity'])
        
        perf_row = {'Model': name}
        perf_row['AUC_Mean'], perf_row['AUC_SD'] = np.mean(auc_scores), np.std(auc_scores)
        perf_row['AUC_95CI_lower'] = perf_row['AUC_Mean'] - 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        perf_row['AUC_95CI_upper'] = perf_row['AUC_Mean'] + 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        
        perf_row['Sensitivity_Mean'], perf_row['Sensitivity_SD'] = np.mean(sens_scores), np.std(sens_scores)
        perf_row['Sensitivity_95CI_lower'] = perf_row['Sensitivity_Mean'] - 1.96 * (perf_row['Sensitivity_SD'] / np.sqrt(len(sens_scores)))
        perf_row['Sensitivity_95CI_upper'] = perf_row['Sensitivity_Mean'] + 1.96 * (perf_row['Sensitivity_SD'] / np.sqrt(len(sens_scores)))
        
        perf_row['Specificity_Mean'], perf_row['Specificity_SD'] = np.mean(spec_scores), np.std(spec_scores)
        perf_row['Specificity_95CI_lower'] = perf_row['Specificity_Mean'] - 1.96 * (perf_row['Specificity_SD'] / np.sqrt(len(spec_scores)))
        perf_row['Specificity_95CI_upper'] = perf_row['Specificity_Mean'] + 1.96 * (perf_row['Specificity_SD'] / np.sqrt(len(spec_scores)))
        
        perf_row['Avg_Threshold'] = np.mean(thresholds)
        performance_list.append(perf_row)
        print(f"  Metrics: AUC={perf_row['AUC_Mean']:.3f}, Sens={perf_row['Sensitivity_Mean']:.3f}, Spec={perf_row['Specificity_Mean']:.3f}")

    # --- Save and Plot Final Results ---
    params_df = pd.DataFrame(best_params_list)
    performance_df = pd.DataFrame(performance_list)
    params_df.to_excel(OUTPUT_PARAMS_EXCEL, index=False)
    performance_df.to_excel(OUTPUT_PERF_EXCEL, index=False)
    print(f"\nSaved results to '{OUTPUT_FOLDER}' folder.")
    
    print("Generating final plot...")
    model_order = ['DT', 'KNN', 'NNET', 'NB', 'SVM', 'SGBT', 'LR', 'RF', 'XGB']
    performance_df['Model'] = pd.Categorical(performance_df['Model'], categories=model_order, ordered=True)
    performance_df.sort_values('Model', inplace=True)
    metrics_to_plot = {'ROC': 'AUC_Mean', 'Spec': 'Specificity_Mean', 'Sens': 'Sensitivity_Mean'}
    
    fig, axes = plt.subplots(1, 3, figsize=(22, 10))
    for ax, (plot_name, metric_col) in zip(axes, metrics_to_plot.items()):
        vals = performance_df[metric_col]
        y_pos = np.arange(len(performance_df['Model']))
        ax.barh(y_pos, vals, color='skyblue', edgecolor='black', height=0.6)
        ax.set_yticks(y_pos); ax.set_yticklabels(performance_df['Model'], fontsize=12); ax.invert_yaxis()
        ax.set_title(plot_name, fontsize=16); ax.set_xlabel('Value', fontsize=14); ax.set_xlim(0, 1.15)
        for i, v in enumerate(vals):
            ax.text(v + 0.02, i, f"{v:.3f}", va='center', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURE_FILE, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to '{OUTPUT_FIGURE_FILE}'")
    print("\nProcess finished successfully.")

if __name__ == '__main__':
    main()

Current working directory: /mnt/d/AKI新分析
--- Data Loading (Simplified) ---
Loading data: train_imputed_random_forest.csv
  > Target 'PostopAKI' loaded. Positive rate: 0.0298
Data ready. Features: 15, Samples: 2446

--- Processing Model: LR ---

--- STAGE 1: Coarse Grid Search for LR ---
Fitting 5 folds for each of 3 candidates, totalling 15 fits


/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

  Coarse search best params: {'classifier__C': 0.01}

--- STAGE 2: Focused Bayesian Search for LR ---
Fitting 100 folds for each of 1 candidates, totalling 100 fits


/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 1

...

  Coarse search best params: {'classifier__n_neighbors': 15}

--- STAGE 2: Focused Bayesian Search for KNN ---
Fitting 100 folds for each of 1 candidates, totalling 100 fits


/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 1

/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 100 fits
Fitting 100 folds for each of 1 candidates, totalling 1

In [ ]:
# -*- coding: utf-8 -*-
# 文件名: run_hyperopt_search_rf_simple_fixed.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import random
import ast
from pathlib import Path

# Set global random seeds for maximum reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, roc_auc_score, confusion_matrix, roc_curve, accuracy_score, f1_score, recall_score, precision_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import BayesianRidge

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Configuration ---
# Hyperopt 主要是串行优化，内部模型并行设为 -1 (使用所有核心)
N_JOBS_MODEL_INTERNAL = -1 
BASE_DIR = Path(os.getcwd())
RFE_LISTS_FOLDER = BASE_DIR / 'RFE_Final_Run/feature_lists'

# 🌟 修改点 1: 指定包含 PostopAKI 的插补文件 🌟
IMPUTED_DATA_FILE = BASE_DIR / 'imputation' / 'imputed_data' / 'train_imputed_random_forest.csv'
TARGET_VARIABLE = 'PostopAKI'

OUTPUT_FOLDER = BASE_DIR / 'Tuning_Hyperopt_Results'
DATA_OUTPUT_FOLDER = OUTPUT_FOLDER / 'final_training_data'
OUTPUT_PARAMS_EXCEL = OUTPUT_FOLDER / 'best_hyperparameters_hyperopt.xlsx'
OUTPUT_PERF_EXCEL = OUTPUT_FOLDER / 'model_performance_hyperopt.xlsx'
OUTPUT_FIGURE_FILE = OUTPUT_FOLDER / 'internal_validation_plot_hyperopt.png'
MAX_EVALS_HYPEROPT = 50

# --- Helper Functions ---
def specificity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tn / (tn + fp) if (tn + fp) > 0 else 0.0
def sensitivity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def find_optimal_threshold(y_true, y_prob):
    """Find the threshold that maximizes Youden's Index."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden_index = tpr - fpr
    best_idx = np.argmax(youden_index)
    return thresholds[best_idx]

def get_metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

# --- Main Logic ---
def main():
    for folder in [OUTPUT_FOLDER, DATA_OUTPUT_FOLDER]: folder.mkdir(parents=True, exist_ok=True)

    print("--- Data Loading (Simplified) ---")
    
    # 🌟 1. 直接读取数据 🌟
    print(f"Loading data: {IMPUTED_DATA_FILE.name}")
    try:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='gbk')
    except UnicodeDecodeError:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='utf-8')

    if 'Unnamed: 0' in df_train.columns: df_train.drop(columns=['Unnamed: 0'], inplace=True)

    # 🌟 2. 提取 Target 🌟
    if TARGET_VARIABLE not in df_train.columns:
        raise ValueError(f"CRITICAL ERROR: '{TARGET_VARIABLE}' not found in imputed file.")
    
    y_train = df_train[TARGET_VARIABLE].astype(int)
    print(f"  > Target '{TARGET_VARIABLE}' loaded. Positive rate: {y_train.mean():.4f}")

    # 🌟 3. 特征准备 (使用白名单机制) 🌟
    preoperative_vars = [ 'Gender', 'Age', 'Height', 'Weight', 'BMI', 'PreopHospitalDays', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'PreopWBC', 'PreopHb', 'PreopAlb', 'PreopALT', 'PreopBUN', 'PreopCr', 'PreopGlucose' ]
    intraoperative_vars = [ 'SurgicalApproach', 'OperationTime', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopBloodLoss', 'IntraopTransfusion', 'IntraopFluid', 'IntraopHES', 'IntraopDextran', 'IntraopGelatin', 'IntraopPlasma', 'IntraopColloid', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage', 'LymphNodesExamined', 'PositiveLymphNodes' ]
    all_predictors_base = preoperative_vars + intraoperative_vars
    categorical_vars_base = ['Gender', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'SurgicalApproach', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopTransfusion', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage']
    
    current_predictors = [v for v in all_predictors_base if v in df_train.columns]
    current_categoricals = [v for v in categorical_vars_base if v in df_train.columns]
    current_continuous = [v for v in current_predictors if v not in current_categoricals]
    
    # 清洗连续变量
    for col in current_continuous:
        df_train[col] = df_train[col].astype(str).str.extract(r'^(\d+[,.]?\d*)', expand=False)
        df_train[col] = df_train[col].str.replace(',', '.', regex=False)
        df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
    
    # 计算衍生指标
    required_cols = ['PLT', 'LYM', 'MONO', 'NE', 'EOS', 'BASO']
    if all(col in df_train.columns for col in required_cols):
        print("  Calculating derived features (NLR, PLR, etc.)...")
        for col in ['LYM']: df_train[col] = df_train[col].replace(0, 1e-6)
        new_features = ['PLR', 'MLR', 'ELR', 'BLR', 'NLR', 'SII']
        df_train['PLR'], df_train['MLR'], df_train['ELR'], df_train['BLR'], df_train['NLR'], df_train['SII'] = \
            df_train['PLT']/df_train['LYM'], df_train['MONO']/df_train['LYM'], df_train['EOS']/df_train['LYM'], \
            df_train['BASO']/df_train['LYM'], df_train['NE']/df_train['LYM'], df_train['PLT']*df_train['NE']/df_train['LYM']
        current_predictors.extend([f for f in new_features if f not in current_predictors])
    
    # 🌟 4. 构建特征矩阵 X 🌟
    # 只包含 current_predictors 列表中的列，绝对安全！
    X_train_full_raw = df_train[current_predictors]
    X_train_full_dummies = pd.get_dummies(X_train_full_raw, columns=current_categoricals, drop_first=True)
    X_train_imputed_full = X_train_full_dummies.copy()
    
    if X_train_imputed_full.isnull().sum().sum() > 0:
        X_train_imputed_full.fillna(X_train_imputed_full.median(), inplace=True)

    print(f"Data ready. Features: {len(X_train_imputed_full.columns)}, Samples: {len(X_train_imputed_full)}")
    
    # --- Define Models and Hyperparameter Search Spaces for Hyperopt ---
    models_and_spaces = {
        'LR': (LogisticRegression(solver='liblinear', random_state=SEED, max_iter=2000), 
               {'classifier__penalty': hp.choice('lr_penalty', ['l1', 'l2']), 'classifier__C': hp.loguniform('lr_C', np.log(0.001), np.log(100))}),
        'DT': (DecisionTreeClassifier(random_state=SEED),
               {'classifier__max_depth': hp.choice('dt_max_depth', [None] + list(range(5, 50))), 'classifier__min_samples_split': hp.quniform('dt_min_samples_split', 2, 20, 1)}),
        'RF': (RandomForestClassifier(random_state=SEED),
               {'classifier__n_estimators': hp.quniform('rf_n_estimators', 100, 500, 50), 'classifier__max_depth': hp.choice('rf_max_depth', [None] + list(range(10, 60, 10)))}),
        'KNN': (KNeighborsClassifier(),
                {'classifier__n_neighbors': hp.quniform('knn_n_neighbors', 3, 25, 2), 'classifier__metric': hp.choice('knn_metric', ['euclidean', 'manhattan'])}),
        'SVM': (SVC(probability=True, random_state=SEED),
                {'classifier__C': hp.loguniform('svm_C', np.log(0.1), np.log(100)), 'classifier__gamma': hp.loguniform('svm_gamma', np.log(0.0001), np.log(0.1))}),
        'NB': (GaussianNB(),
               {'classifier__var_smoothing': hp.loguniform('nb_var_smoothing', np.log(1e-10), np.log(1e-1))}),
        'XGB': (XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED),
                {'classifier__n_estimators': hp.quniform('xgb_n_estimators', 100, 400, 50), 'classifier__learning_rate': hp.loguniform('xgb_learning_rate', np.log(0.01), np.log(0.3)), 'classifier__max_depth': hp.quniform('xgb_max_depth', 3, 10, 1)}),
        'SGBT': (GradientBoostingClassifier(random_state=SEED),
                 {'classifier__n_estimators': hp.quniform('sgbt_n_estimators', 100, 400, 50), 'classifier__learning_rate': hp.loguniform('sgbt_learning_rate', np.log(0.01), np.log(0.3)), 'classifier__max_depth': hp.quniform('sgbt_max_depth', 3, 10, 1)}),
        'NNET': (MLPClassifier(max_iter=500, random_state=SEED, early_stopping=True),
                 {'classifier__hidden_layer_sizes': hp.choice('nnet_hidden_layer_sizes', [(50,), (100,), (50, 50), (100, 50)]), 'classifier__alpha': hp.loguniform('nnet_alpha', np.log(1e-5), np.log(1e-2))})
    }

    cv_full = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=SEED)
    best_params_list, performance_list = [], []
    current_n_jobs = -1

    for name, (model, space) in models_and_spaces.items():
        print(f"\n--- Processing Model: {name} ---")
        feature_file_path = RFE_LISTS_FOLDER / f'selected_features_{name}_Combined.txt'
        try:
            with open(feature_file_path, 'r') as f: selected_features = [line.strip() for line in f if not line.strip().startswith('#') and line.strip()]
        except FileNotFoundError: print(f"  Warning: Feature file not found. Skipping."); continue
            
        valid_features = [f for f in selected_features if f in X_train_imputed_full.columns]
        if not valid_features:
            print("  Warning: No valid features found for this model.")
            continue
            
        X_train_model_specific = X_train_imputed_full[valid_features]
        pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])

        # Define objective for Hyperopt
        def objective(params):
            # Hyperopt returns floats for quniform, sklearn needs ints for these params
            for param in ['classifier__n_estimators', 'classifier__max_depth', 'classifier__min_child_weight', 'classifier__n_neighbors', 'classifier__min_samples_split']:
                if param in params and params[param] is not None:
                    params[param] = int(params[param])
            
            pipeline.set_params(**params)
            # Use 5-fold CV for speed during search
            score = cross_val_score(pipeline, X_train_model_specific, y_train, scoring='roc_auc', cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=SEED), n_jobs=-1).mean()
            return {'loss': 1 - score, 'status': STATUS_OK}

        print(f"  Starting Hyperopt Search for {name} ({MAX_EVALS_HYPEROPT} evals)...")
        trials = Trials()
        best_params_raw = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=MAX_EVALS_HYPEROPT, trials=trials, rstate=np.random.default_rng(SEED))
        best_params = space_eval(space, best_params_raw)
        
        # Clean params (convert back to int for final usage)
        for param in ['classifier__n_estimators', 'classifier__max_depth', 'classifier__n_neighbors', 'classifier__min_samples_split']:
            if param in best_params and best_params[param] is not None: 
                best_params[param] = int(best_params[param])
            
        best_params_list.append({'Model': name, 'Best Hyperparameters': str(best_params)})
        
        # --- 🌟 CRITICAL: Find Optimal Threshold using Cross-Validation ---
        print(f"  Calculating performance metrics with Optimal Threshold...")
        final_pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])
        final_pipeline.set_params(**best_params)
        
        auc_scores, sens_scores, spec_scores, thresholds = [], [], [], []
        
        # Manual CV loop for stable metric calculation
        for i, (train_idx, val_idx) in enumerate(cv_full.split(X_train_model_specific, y_train)):
            X_train_fold, X_val_fold = X_train_model_specific.iloc[train_idx], X_train_model_specific.iloc[val_idx]
            y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            final_pipeline.fit(X_train_fold, y_train_fold)
            y_prob_val = final_pipeline.predict_proba(X_val_fold)[:, 1]
            best_thresh = find_optimal_threshold(y_val_fold, y_prob_val)
            thresholds.append(best_thresh)
            
            metrics = get_metrics_at_threshold(y_val_fold, y_prob_val, best_thresh)
            auc_scores.append(roc_auc_score(y_val_fold, y_prob_val))
            sens_scores.append(metrics['Sensitivity'])
            spec_scores.append(metrics['Specificity'])
        
        perf_row = {'Model': name}
        perf_row['AUC_Mean'], perf_row['AUC_SD'] = np.mean(auc_scores), np.std(auc_scores)
        perf_row['AUC_95CI_lower'] = perf_row['AUC_Mean'] - 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        perf_row['AUC_95CI_upper'] = perf_row['AUC_Mean'] + 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        
        perf_row['Sensitivity_Mean'], perf_row['Sensitivity_SD'] = np.mean(sens_scores), np.std(sens_scores)
        perf_row['Sensitivity_95CI_lower'] = perf_row['Sensitivity_Mean'] - 1.96 * (perf_row['Sensitivity_SD'] / np.sqrt(len(sens_scores)))
        perf_row['Sensitivity_95CI_upper'] = perf_row['Sensitivity_Mean'] + 1.96 * (perf_row['Sensitivity_SD'] / np.sqrt(len(sens_scores)))
        
        perf_row['Specificity_Mean'], perf_row['Specificity_SD'] = np.mean(spec_scores), np.std(spec_scores)
        perf_row['Specificity_95CI_lower'] = perf_row['Specificity_Mean'] - 1.96 * (perf_row['Specificity_SD'] / np.sqrt(len(spec_scores)))
        perf_row['Specificity_95CI_upper'] = perf_row['Specificity_Mean'] + 1.96 * (perf_row['Specificity_SD'] / np.sqrt(len(spec_scores)))
        
        perf_row['Avg_Threshold'] = np.mean(thresholds)
        performance_list.append(perf_row)
        print(f"  Metrics: AUC={perf_row['AUC_Mean']:.3f}, Sens={perf_row['Sensitivity_Mean']:.3f}, Spec={perf_row['Specificity_Mean']:.3f}")

    # --- Save and Plot Final Results ---
    params_df = pd.DataFrame(best_params_list)
    performance_df = pd.DataFrame(performance_list)
    params_df.to_excel(OUTPUT_PARAMS_EXCEL, index=False)
    performance_df.to_excel(OUTPUT_PERF_EXCEL, index=False)
    print(f"\nSaved results to '{OUTPUT_FOLDER}' folder.")
    
    # 绘图逻辑 (修正 Key Error)
    print("Generating final plot...")
    model_order = ['DT', 'KNN', 'NNET', 'NB', 'SVM', 'SGBT', 'LR', 'RF', 'XGB']
    performance_df['Model'] = pd.Categorical(performance_df['Model'], categories=model_order, ordered=True)
    performance_df.sort_values('Model', inplace=True)
    
    # 🌟 修正字典值：去掉 _Mean 后缀，因为下面代码会自动加 🌟
    metrics_to_plot = {'ROC': 'AUC', 'Spec': 'Specificity', 'Sens': 'Sensitivity'}
    
    fig, axes = plt.subplots(1, 3, figsize=(22, 10))
    for ax, (plot_name, metric_base) in zip(axes, metrics_to_plot.items()):
        means = performance_df[f'{metric_base}_Mean'] # 拼接成 'AUC_Mean'
        sds = performance_df[f'{metric_base}_SD']
        ci_lowers, ci_uppers = performance_df[f'{metric_base}_95CI_lower'], performance_df[f'{metric_base}_95CI_upper']
        y_pos = np.arange(len(performance_df['Model']))
        ci_error, sd_error = [means - ci_lowers, ci_uppers - means], sds
        ax.errorbar(means, y_pos, xerr=ci_error, fmt='none', ecolor='skyblue', elinewidth=10, capsize=0)
        ax.errorbar(means, y_pos, xerr=sd_error, fmt='none', ecolor='dimgray', elinewidth=1.5, capsize=4)
        ax.plot(means, y_pos, 'o', color='black', markersize=6)
        ax.set_yticks(y_pos); ax.set_yticklabels(performance_df['Model'], fontsize=12); ax.invert_yaxis()
        ax.set_title(plot_name, fontsize=16); ax.set_xlabel('Value', fontsize=14)
        ax.set_xlim(0, 1.2); ax.grid(axis='x', linestyle='--', alpha=0.6)
        for i in range(len(performance_df)):
            if i < len(performance_df) and i < len(means):
                mean_val, sd_val = means.iloc[i], sds.iloc[i]
                ci_lower_val, ci_upper_val = ci_lowers.iloc[i], ci_uppers.iloc[i]
                try:
                    ax.text(1.03, y_pos[i] + 0.18, f"{mean_val:.3f}±{sd_val:.3f}", va='center', ha='left', fontsize=11)
                    ax.text(1.03, y_pos[i] - 0.18, f"95% CI ({ci_lower_val:.4f}‒{ci_upper_val:.4f})", va='center', ha='left', fontsize=11)
                except Exception: pass
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURE_FILE, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to '{OUTPUT_FIGURE_FILE}'")
    print("\nProcess finished successfully.")

if __name__ == '__main__':
    main()

--- Data Loading (Simplified) ---
Loading data: train_imputed_random_forest.csv
  > Target 'PostopAKI' loaded. Positive rate: 0.0298
Data ready. Features: 15, Samples: 2446

--- Processing Model: LR ---
  Starting Hyperopt Search for LR (50 evals)...
  0%|                                                                            | 0/50 [00:00<?, ?trial/s, best loss=?]

/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

  2%|▉                                                | 1/50 [00:09<07:39,  9.39s/trial, best loss: 0.21957361758827454]

/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

  4%|█▉                                               | 2/50 [00:11<03:55,  4.91s/trial, best loss: 0.21957361758827454]

...

 16%|████████▏                                          | 8/50 [00:14<01:15,  1.79s/trial, best loss: 0.359363125112359]

/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

 18%|████████▊                                        | 9/50 [00:16<01:12,  1.78s/trial, best loss: 0.35929751805674537]

/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/blanch/miniconda3/envs/machinelearn/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11

100%|████████████████████████████████████████████████| 50/50 [00:19<00:00,  2.59trial/s, best loss: 0.34669137823461615]
  Calculating performance metrics with Optimal Threshold...
  Metrics: AUC=0.637, Sens=0.681, Spec=0.678

--- Processing Model: XGB ---
  Starting Hyperopt Search for XGB (50 evals)...
100%|████████████████████████████████████████████████| 50/50 [00:26<00:00,  1.91trial/s, best loss: 0.24903659147869672]
  Calculating performance metrics with Optimal Threshold...
  Metrics: AUC=0.749, Sens=0.823, Spec=0.685

--- Processing Model: SGBT ---
  Starting Hyperopt Search for SGBT (50 evals)...
100%|█████████████████████████████████████████████████| 50/50 [03:24<00:00,  4.09s/trial, best loss: 0.2680345335913624]
  Calculating performance metrics with Optimal Threshold...


In [2]:
# -*- coding: utf-8 -*-
# 文件名: run_grid_search_rf_simple.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import random
import ast
from pathlib import Path

# Set global random seeds
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, roc_auc_score, confusion_matrix, roc_curve, accuracy_score, f1_score, recall_score, precision_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import BayesianRidge
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.base import clone

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Configuration ---
N_JOBS_MODEL_INTERNAL = 40
BASE_DIR = Path(os.getcwd())
RFE_LISTS_FOLDER = BASE_DIR / 'RFE_Final_Run/feature_lists'

# 🌟 修改点 1: 指定包含 PostopAKI 的插补文件 🌟
IMPUTED_DATA_FILE = BASE_DIR / 'imputation' / 'imputed_data' / 'train_imputed_random_forest.csv'
TARGET_VARIABLE = 'PostopAKI'

OUTPUT_FOLDER = BASE_DIR / 'Tuning_GridSearch_Results'
DATA_OUTPUT_FOLDER = OUTPUT_FOLDER / 'final_training_data'
OUTPUT_PARAMS_EXCEL = OUTPUT_FOLDER / 'best_hyperparameters_grid.xlsx'
OUTPUT_PERF_EXCEL = OUTPUT_FOLDER / 'model_performance_grid.xlsx'
OUTPUT_FIGURE_FILE = OUTPUT_FOLDER / 'internal_validation_plot_grid.png'

# --- Helper Functions ---
def specificity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tn / (tn + fp) if (tn + fp) > 0 else 0.0
def sensitivity_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel(); return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def find_optimal_threshold(y_true, y_prob):
    """Find the threshold that maximizes Youden's Index."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden_index = tpr - fpr
    best_idx = np.argmax(youden_index)
    return thresholds[best_idx]

def get_metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

# --- Main Logic ---
def main():
    for folder in [OUTPUT_FOLDER, DATA_OUTPUT_FOLDER]: Path(folder).mkdir(parents=True, exist_ok=True)

    print("--- Data Loading (Simplified) ---")
    
    # 🌟 1. 直接读取数据 🌟
    print(f"Loading data: {IMPUTED_DATA_FILE.name}")
    try:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='gbk')
    except UnicodeDecodeError:
        df_train = pd.read_csv(IMPUTED_DATA_FILE, encoding='utf-8')

    # 清洗无名索引列
    if 'Unnamed: 0' in df_train.columns: df_train.drop(columns=['Unnamed: 0'], inplace=True)

    # 🌟 2. 提取 Target 🌟
    if TARGET_VARIABLE not in df_train.columns:
        raise ValueError(f"CRITICAL ERROR: '{TARGET_VARIABLE}' not found in imputed file.")
    
    y_train = df_train[TARGET_VARIABLE].astype(int)
    print(f"  > Target '{TARGET_VARIABLE}' loaded. Positive rate: {y_train.mean():.4f}")

    # 🌟 3. 特征准备 (使用白名单机制) 🌟
    preoperative_vars = [ 'Gender', 'Age', 'Height', 'Weight', 'BMI', 'PreopHospitalDays', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'PreopWBC', 'PreopHb', 'PreopAlb', 'PreopALT', 'PreopBUN', 'PreopCr', 'PreopGlucose' ]
    intraoperative_vars = [ 'SurgicalApproach', 'OperationTime', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopBloodLoss', 'IntraopTransfusion', 'IntraopFluid', 'IntraopHES', 'IntraopDextran', 'IntraopGelatin', 'IntraopPlasma', 'IntraopColloid', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage', 'LymphNodesExamined', 'PositiveLymphNodes' ]
    all_predictors_base = preoperative_vars + intraoperative_vars
    categorical_vars_base = ['Gender', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'SurgicalApproach', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopTransfusion', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage']
    
    current_predictors = [v for v in all_predictors_base if v in df_train.columns]
    current_categoricals = [v for v in categorical_vars_base if v in df_train.columns]
    current_continuous = [v for v in current_predictors if v not in current_categoricals]
    
    # 清洗连续变量
    for col in current_continuous:
        df_train[col] = df_train[col].astype(str).str.extract(r'^(\d+[,.]?\d*)', expand=False)
        df_train[col] = df_train[col].str.replace(',', '.', regex=False)
        df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
    
    # 计算衍生指标
    required_cols = ['PLT', 'LYM', 'MONO', 'NE', 'EOS', 'BASO']
    if all(col in df_train.columns for col in required_cols):
        print("  Calculating derived features (NLR, PLR, etc.)...")
        for col in ['LYM']: df_train[col] = df_train[col].replace(0, 1e-6)
        new_features = ['PLR', 'MLR', 'ELR', 'BLR', 'NLR', 'SII']
        df_train['PLR'], df_train['MLR'], df_train['ELR'], df_train['BLR'], df_train['NLR'], df_train['SII'] = \
            df_train['PLT']/df_train['LYM'], df_train['MONO']/df_train['LYM'], df_train['EOS']/df_train['LYM'], \
            df_train['BASO']/df_train['LYM'], df_train['NE']/df_train['LYM'], df_train['PLT']*df_train['NE']/df_train['LYM']
        current_predictors.extend([f for f in new_features if f not in current_predictors])
    
    # 🌟 4. 构建特征矩阵 X 🌟
    # 只包含 current_predictors 列表中的列，绝对安全！
    X_train_full_raw = df_train[current_predictors]
    X_train_full_dummies = pd.get_dummies(X_train_full_raw, columns=current_categoricals, drop_first=True)
    X_train_imputed_full = X_train_full_dummies.copy()
    
    if X_train_imputed_full.isnull().sum().sum() > 0:
        X_train_imputed_full.fillna(X_train_imputed_full.median(), inplace=True)

    print(f"Data ready. Features: {len(X_train_imputed_full.columns)}")

    # --- Models ---
    models = {
        'LR': LogisticRegression(solver='liblinear', random_state=SEED, max_iter=2000),
        'DT': DecisionTreeClassifier(random_state=SEED),
        'RF': RandomForestClassifier(random_state=SEED, n_jobs=N_JOBS_MODEL_INTERNAL),
        'KNN': KNeighborsClassifier(n_jobs=N_JOBS_MODEL_INTERNAL),
        'SVM': SVC(probability=True, random_state=SEED),
        'NB': GaussianNB(),
        'XGB': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED, n_jobs=N_JOBS_MODEL_INTERNAL),
        'SGBT': GradientBoostingClassifier(random_state=SEED),
        'NNET': MLPClassifier(max_iter=500, random_state=SEED, early_stopping=True)
    }

    # --- Parameter Grids ---
    param_grids = {
        'LR': {'classifier__C': [0.01, 0.1, 1, 10, 100], 'classifier__penalty': ['l1', 'l2']},
        'DT': {'classifier__max_depth': [5, 10, 20, None], 'classifier__min_samples_split': [2, 5, 10]},
        'RF': {'classifier__n_estimators': [100, 300], 'classifier__max_depth': [10, 30, None]},
        'KNN': {'classifier__n_neighbors': [5, 10, 20], 'classifier__weights': ['uniform', 'distance']},
        'SVM': {'classifier__C': [0.1, 1, 10], 'classifier__gamma': [0.001, 0.01, 0.1, 'scale']},
        'NB': {'classifier__var_smoothing': [1e-9, 1e-7, 1e-5]},
        'XGB': {'classifier__n_estimators': [100, 300], 'classifier__learning_rate': [0.01, 0.1, 0.2], 'classifier__max_depth': [3, 6]},
        'SGBT': {'classifier__n_estimators': [100, 300], 'classifier__learning_rate': [0.01, 0.1, 0.2], 'classifier__max_depth': [3, 6]},
        'NNET': {'classifier__hidden_layer_sizes': [(50,), (100,), (50, 50)], 'classifier__alpha': [0.0001, 0.001, 0.01]}
    }

    cv_search = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=SEED)
    cv_full = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=SEED)
    
    best_params_list, performance_list = [], []

    for name, model in models.items():
        print(f"\n--- Processing Model: {name} ---")
        feature_file = RFE_LISTS_FOLDER / f'selected_features_{name}_Combined.txt'
        try:
            with open(feature_file, 'r') as f: 
                selected_features = [line.strip() for line in f if not line.strip().startswith('#') and line.strip()]
        except FileNotFoundError: 
            print(f"  Warning: Feature file not found at {feature_file}. Skipping.")
            continue
            
        valid_features = [f for f in selected_features if f in X_train_imputed_full.columns]
        X_train_model_specific = X_train_imputed_full[valid_features]
        
        pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])

        if name in ['XGB', 'RF', 'KNN']: search_n_jobs = 1
        elif name in ['NNET', 'SVM', 'SGBT']: search_n_jobs = 8
        else: search_n_jobs = 30

        print(f"  Starting GridSearchCV (n_jobs={search_n_jobs})...")
        grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grids[name], scoring='roc_auc', cv=cv_search, n_jobs=search_n_jobs, verbose=1)
        grid_search.fit(X_train_model_specific, y_train)
        
        best_params_recorded = grid_search.best_params_
        best_params_list.append({'Model': name, 'Best Hyperparameters': str(best_params_recorded)})
        
        # --- Metrics Calculation ---
        print(f"  Calculating performance metrics with Optimal Threshold...")
        final_pipeline = grid_search.best_estimator_
        auc_scores, sens_scores, spec_scores, thresholds = [], [], [], []
        
        for i, (train_idx, val_idx) in enumerate(cv_full.split(X_train_model_specific, y_train)):
            X_train_fold, X_val_fold = X_train_model_specific.iloc[train_idx], X_train_model_specific.iloc[val_idx]
            y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            final_pipeline.fit(X_train_fold, y_train_fold)
            y_prob_val = final_pipeline.predict_proba(X_val_fold)[:, 1]
            best_thresh = find_optimal_threshold(y_val_fold, y_prob_val)
            thresholds.append(best_thresh)
            
            metrics = get_metrics_at_threshold(y_val_fold, y_prob_val, best_thresh)
            auc_scores.append(roc_auc_score(y_val_fold, y_prob_val))
            sens_scores.append(metrics['Sensitivity'])
            spec_scores.append(metrics['Specificity'])
        
        perf_row = {'Model': name}
        perf_row['AUC_Mean'], perf_row['AUC_SD'] = np.mean(auc_scores), np.std(auc_scores)
        perf_row['AUC_95CI_lower'] = perf_row['AUC_Mean'] - 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        perf_row['AUC_95CI_upper'] = perf_row['AUC_Mean'] + 1.96 * (perf_row['AUC_SD'] / np.sqrt(len(auc_scores)))
        
        perf_row['Sensitivity_Mean'], perf_row['Sensitivity_SD'] = np.mean(sens_scores), np.std(sens_scores)
        perf_row['Sensitivity_95CI_lower'] = perf_row['Sensitivity_Mean'] - 1.96 * (perf_row['Sensitivity_SD'] / np.sqrt(len(sens_scores)))
        perf_row['Sensitivity_95CI_upper'] = perf_row['Sensitivity_Mean'] + 1.96 * (perf_row['Sensitivity_SD'] / np.sqrt(len(sens_scores)))
        
        perf_row['Specificity_Mean'], perf_row['Specificity_SD'] = np.mean(spec_scores), np.std(spec_scores)
        perf_row['Specificity_95CI_lower'] = perf_row['Specificity_Mean'] - 1.96 * (perf_row['Specificity_SD'] / np.sqrt(len(spec_scores)))
        perf_row['Specificity_95CI_upper'] = perf_row['Specificity_Mean'] + 1.96 * (perf_row['Specificity_SD'] / np.sqrt(len(spec_scores)))
        
        perf_row['Avg_Threshold'] = np.mean(thresholds)
        performance_list.append(perf_row)
        print(f"  Metrics: AUC={perf_row['AUC_Mean']:.3f}, Sens={perf_row['Sensitivity_Mean']:.3f}, Spec={perf_row['Specificity_Mean']:.3f}")

    # --- Save and Plot ---
    params_df = pd.DataFrame(best_params_list)
    performance_df = pd.DataFrame(performance_list)
    params_df.to_excel(OUTPUT_PARAMS_EXCEL, index=False)
    performance_df.to_excel(OUTPUT_PERF_EXCEL, index=False)
    print(f"\nSaved results to '{OUTPUT_FOLDER}' folder.")
    
    print("Generating final plot...")
    model_order = ['DT', 'KNN', 'NNET', 'NB', 'SVM', 'SGBT', 'LR', 'RF', 'XGB']
    performance_df['Model'] = pd.Categorical(performance_df['Model'], categories=model_order, ordered=True)
    performance_df.sort_values('Model', inplace=True)
    metrics_to_plot = {'ROC': 'AUC_Mean', 'Spec': 'Specificity_Mean', 'Sens': 'Sensitivity_Mean'}
    fig, axes = plt.subplots(1, 3, figsize=(22, 10))
    for ax, (plot_name, metric_col) in zip(axes, metrics_to_plot.items()):
        vals = performance_df[metric_col]
        y_pos = np.arange(len(performance_df['Model']))
        ax.barh(y_pos, vals, color='skyblue', edgecolor='black', height=0.6)
        ax.set_yticks(y_pos); ax.set_yticklabels(performance_df['Model'], fontsize=12); ax.invert_yaxis()
        ax.set_title(plot_name, fontsize=16); ax.set_xlabel('Value', fontsize=14); ax.set_xlim(0, 1.15)
        for i, v in enumerate(vals):
            ax.text(v + 0.02, i, f"{v:.3f}", va='center', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURE_FILE, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to '{OUTPUT_FIGURE_FILE}'")
    print("\nProcess finished successfully.")

if __name__ == '__main__':
    main()

--- Data Loading (Simplified) ---
Loading data: train_imputed_random_forest.csv
  > Target 'PostopAKI' loaded. Positive rate: 0.0298
Data ready. Features: 15

--- Processing Model: LR ---
  Starting GridSearchCV (n_jobs=30)...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
  Calculating performance metrics with Optimal Threshold...
  Metrics: AUC=0.768, Sens=0.766, Spec=0.775

--- Processing Model: DT ---
  Starting GridSearchCV (n_jobs=30)...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
  Calculating performance metrics with Optimal Threshold...
  Metrics: AUC=0.663, Sens=0.702, Spec=0.672

--- Processing Model: RF ---
  Starting GridSearchCV (n_jobs=1)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
  Calculating performance metrics with Optimal Threshold...
  Metrics: AUC=0.785, Sens=0.850, Spec=0.706

--- Processing Model: KNN ---
  Starting GridSearchCV (n_jobs=1)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
  Calculat

In [1]:
# -*- coding: utf-8 -*-
# 文件名: evaluate_all_methods.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# --- 1. 配置：对应您运行的五个脚本的输出文件夹 ---
# 请确保这些文件夹名称与之前脚本生成的完全一致
FOLDERS = {
    'Grid Search': 'Tuning_GridSearch_Results',
    'Optuna': 'Tuning_Optuna_Results',
    'Two-Stage': 'Tuning_TwoStage_Results',
    'Hyperopt': 'Tuning_Hyperopt_Results',
    'Scikit-Optimize': 'Tuning_BayesSearch_Results_Layered' 
}

# --- 2. 对应的结果文件名 ---
FILE_NAMES = {
    'Grid Search': 'model_performance_grid.xlsx',
    'Optuna': 'model_performance_optuna.xlsx',
    'Two-Stage': 'model_performance_twostage.xlsx',
    'Hyperopt': 'model_performance_hyperopt.xlsx',
    'Scikit-Optimize': 'model_performance_bayes.xlsx'
}

# 定义输出文件名
OUTPUT_SUMMARY_EXCEL = "All_Methods_Comparison_Summary.xlsx"
OUTPUT_COMPARISON_PLOT = "Optimization_Methods_Comparison_Plot.png"
OUTPUT_FOREST_PLOT = "Best_Method_Forest_Plot.png"

def main():
    all_results = []
    print(f"{'='*60}")
    print(f"{'正在读取所有方法的运行结果...':^60}")
    print(f"{'='*60}")
    
    # --- 读取数据 ---
    for method_name, folder in FOLDERS.items():
        file_name = FILE_NAMES.get(method_name)
        if not file_name: continue
            
        file_path = os.path.join(folder, file_name)
        
        if os.path.exists(file_path):
            try:
                df = pd.read_excel(file_path)
                df['Method'] = method_name # 添加方法列
                
                # 确保必要的列存在，若不存在则补全 (防止某些脚本版本不一致)
                # 我们现在的脚本都生成了 _Mean, _SD, _95CI_lower, _95CI_upper
                required_cols = ['AUC_Mean', 'AUC_SD', 'Sensitivity_Mean', 'Specificity_Mean']
                for col in required_cols:
                    if col not in df.columns:
                        df[col] = 0.0 # 缺失则填0，防止报错
                
                all_results.append(df)
                print(f"✅ 成功加载: [{method_name}] - 包含 {len(df)} 个模型")
            except Exception as e:
                print(f"❌ 读取失败: [{method_name}] - 错误: {e}")
        else:
            print(f"⚠️ 未找到文件: {file_path} (该方法可能未运行，跳过)")

    if not all_results:
        print("\n❌ 错误：没有找到任何结果文件。请先运行调参脚本。")
        return

    # 合并所有结果
    final_df = pd.concat(all_results, ignore_index=True)
    
    # --- 1. 找出最佳组合 (基于 AUC_Mean) ---
    if 'AUC_Mean' in final_df.columns:
        best_idx = final_df['AUC_Mean'].idxmax()
        best_row = final_df.loc[best_idx]
        best_method = best_row['Method']
        
        print("\n" + "="*60)
        print(f"🏆 最终大比武结果 (基于 AUC) 🏆".center(60))
        print("="*60)
        print(f"🥇 最佳寻找参数的方法:  【 {best_method} 】")
        print(f"🥇 最佳机器学习模型:    【 {best_row['Model']} 】")
        print("-" * 60)
        print(f"   最佳 AUC (Mean):     {best_row['AUC_Mean']:.4f}")
        print(f"   对应 Sensitivity:    {best_row['Sensitivity_Mean']:.4f}")
        print(f"   对应 Specificity:    {best_row['Specificity_Mean']:.4f}")
        print("="*60)
    else:
        print("Error: 'AUC_Mean' column missing.")
        return
    
    # --- 2. 生成对比表格并保存 ---
    final_df.to_excel(OUTPUT_SUMMARY_EXCEL, index=False)
    print(f"\n已保存详细汇总数据到: {OUTPUT_SUMMARY_EXCEL}")

    # --- 3. 绘制方法对比图 (Bar Plot) ---
    # 这个图展示所有方法下所有模型的 AUC 对比
    plt.figure(figsize=(20, 10)) 
    sns.set(style="whitegrid")
    
    # 绘制分组条形图
    # ci=None 移除误差线，因为这里我们要展示 Mean AUC
    bar = sns.barplot(data=final_df, x='Model', y='AUC_Mean', hue='Method', 
                      palette='viridis', edgecolor='black', ci=None)
    
    plt.title('Comparison of Hyperparameter Tuning Methods (Metric: AUC)', fontsize=20, fontweight='bold')
    plt.xlabel('Machine Learning Model', fontsize=15)
    plt.ylabel('Mean AUC (10x10 CV)', fontsize=15)
    
    # 自动调整Y轴范围，让差异更明显
    y_min = final_df['AUC_Mean'].min()
    y_max = final_df['AUC_Mean'].max()
    plt.ylim(max(0, y_min - 0.1), min(1.0, y_max + 0.05)) 
    
    plt.legend(title='Optimization Method', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_COMPARISON_PLOT, dpi=300)
    plt.close()
    print(f"已保存方法对比图到: {OUTPUT_COMPARISON_PLOT}")

    # --- 4. 绘制最佳方法的详细森林图 (Forest Plot) ---
    print(f"\n正在为最佳方法 【{best_method}】 生成详细森林图...")
    
    # 只筛选出最佳方法的数据来画图
    performance_df = final_df[final_df['Method'] == best_method].copy()
    
    # 定义模型显示顺序
    model_order = ['DT', 'KNN', 'NNET', 'NB', 'SVM', 'SGBT', 'LR', 'RF', 'XGB']
    performance_df['Model'] = pd.Categorical(performance_df['Model'], categories=model_order, ordered=True)
    performance_df.sort_values('Model', inplace=True)
    
    metrics_to_plot = {'ROC': 'AUC', 'Spec': 'Specificity', 'Sens': 'Sensitivity'}
    
    fig, axes = plt.subplots(1, 3, figsize=(24, 10)) # 加宽画布
    
    for ax, (plot_name, metric_base) in zip(axes, metrics_to_plot.items()):
        col_mean = f'{metric_base}_Mean'
        col_sd = f'{metric_base}_SD'
        col_lower = f'{metric_base}_95CI_lower'
        col_upper = f'{metric_base}_95CI_upper'
        
        # 检查列是否存在
        if col_mean not in performance_df.columns:
            print(f"Skipping {metric_base} plot due to missing columns.")
            continue

        means = performance_df[col_mean]
        sds = performance_df[col_sd].fillna(0) # 防止 NaN
        
        # 如果有 CI 数据则使用，否则使用 Mean +/- 1.96*SD 估算
        if col_lower in performance_df.columns and col_upper in performance_df.columns:
            ci_lowers = performance_df[col_lower]
            ci_uppers = performance_df[col_upper]
        else:
            ci_lowers = means - 1.96 * sds
            ci_uppers = means + 1.96 * sds
        
        y_pos = np.arange(len(performance_df['Model']))
        
        # 1. 绘制 95% CI (浅蓝色粗线)
        ci_error = [means - ci_lowers, ci_uppers - means]
        # 处理可能的负误差（当 Mean 接近边界时）
        ci_error = [np.abs(e) for e in ci_error]
        
        ax.errorbar(means, y_pos, xerr=ci_error, fmt='none', ecolor='skyblue', elinewidth=12, capsize=0, label='95% CI')
        
        # 2. 绘制 Mean ± SD (深灰色细线)
        ax.errorbar(means, y_pos, xerr=sds, fmt='none', ecolor='dimgray', elinewidth=2, capsize=5, label='Mean ± SD')
        
        # 3. 绘制均值点 (黑点)
        ax.plot(means, y_pos, 'o', color='black', markersize=8, markeredgecolor='white')
        
        # 设置坐标轴和标题
        ax.set_yticks(y_pos)
        ax.set_yticklabels(performance_df['Model'], fontsize=14, fontweight='bold')
        ax.invert_yaxis()
        ax.set_title(plot_name, fontsize=18, fontweight='bold')
        ax.set_xlabel('Value', fontsize=14)
        ax.set_xlim(0, 1.2) # 留出右侧空间写字
        ax.grid(axis='x', linestyle='--', alpha=0.6)
        
        # 添加数值文本
        for i in range(len(performance_df)):
            if i < len(means):
                mean_val = means.iloc[i]
                sd_val = sds.iloc[i]
                ci_l = ci_lowers.iloc[i]
                ci_u = ci_uppers.iloc[i]
                
                # 第一行：Mean ± SD
                ax.text(1.02, y_pos[i] - 0.15, f"{mean_val:.3f} ± {sd_val:.3f}", 
                        va='center', ha='left', fontsize=11, fontweight='bold', color='black')
                # 第二行：95% CI
                ax.text(1.02, y_pos[i] + 0.15, f"95% CI: {ci_l:.3f}–{ci_u:.3f}", 
                        va='center', ha='left', fontsize=10, color='dimgray')

    plt.tight_layout()
    plt.savefig(OUTPUT_FOREST_PLOT, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"已保存最佳方法森林图到: {OUTPUT_FOREST_PLOT}")
    print("\n🎉 全部完成！请查看生成的 Excel 和 PNG 文件。")

if __name__ == '__main__':
    main()

                      正在读取所有方法的运行结果...                      
✅ 成功加载: [Grid Search] - 包含 9 个模型
⚠️ 未找到文件: Tuning_Optuna_Results/model_performance_optuna.xlsx (该方法可能未运行，跳过)
✅ 成功加载: [Two-Stage] - 包含 9 个模型
✅ 成功加载: [Hyperopt] - 包含 9 个模型
✅ 成功加载: [Scikit-Optimize] - 包含 9 个模型

                    🏆 最终大比武结果 (基于 AUC) 🏆                    
🥇 最佳寻找参数的方法:  【 Two-Stage 】
🥇 最佳机器学习模型:    【 RF 】
------------------------------------------------------------
   最佳 AUC (Mean):     0.7866
   对应 Sensitivity:    0.8518
   对应 Specificity:    0.7073

已保存详细汇总数据到: All_Methods_Comparison_Summary.xlsx
已保存方法对比图到: Optimization_Methods_Comparison_Plot.png

正在为最佳方法 【Two-Stage】 生成详细森林图...
已保存最佳方法森林图到: Best_Method_Forest_Plot.png

🎉 全部完成！请查看生成的 Excel 和 PNG 文件。
